# Kandinsky 2.2 Inpainting — DIMER guided end-to-end notebook (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/kandinsky-inpainting-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/kandinsky-inpainting-pipeline/blob/main/tutorials/kandinsky_inpainting_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-kandinsky--community%2Fkandinsky--2--2--decoder--inpaint-ffcc4d?style=flat)](https://huggingface.co/kandinsky-community/kandinsky-2-2-decoder-inpaint) [![Upstream](https://img.shields.io/badge/Upstream-ai--forever%2FKandinsky--2-181717?style=flat&logo=github&logoColor=white)](https://github.com/ai-forever/Kandinsky-2) [![License](https://img.shields.io/badge/License-Apache--2.0-blue.svg)](https://www.apache.org/licenses/LICENSE-2.0)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** masked image inpainting with a 1.25 B-parameter UNet diffusion model, held-out denoising-loss, preservation and CLIP-scored evaluation, and bounded LoRA fine-tuning to a set of captioned photographs

**This notebook is standalone.** It carries the repository's package (3 modules under `src/kandinsky_inpainting_pipeline/`, at revision `89afe7e81d3b`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `db790ad5cbcabed886f069ef2710774657621702` (~16468 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/3); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh **GPU** runtime (a 16 GB T4 is enough; see the Prerequisites) installs the pinned dependencies, stages and digest-verifies three pinned snapshots from the Hub — the 5.28 GB Kandinsky 2.2 inpainting decoder (UNet + MoVQ), the 10.57 GB Kandinsky 2.2 diffusion prior and a 0.61 GB CLIP scorer — loads the UNet in float16 with an untrained LoRA adapter attached, fetches 60 CC0 iNaturalist bird photographs as digest-verified JPEGs (about 6 MB, no credential), gives each the deterministic centre mask, validates them and splits them 36 / 12 / 12 by seed, encodes every prompt with the prior and releases the prior, measures the frozen model (held-out denoising loss, preservation of the kept region, and CLIP prompt similarity between a mean-colour-fill floor and the original-photograph ceiling), runs a bounded LoRA fine-tuning (4 epochs over 36 photographs), repeats the measurements on identical inputs, inpaints a new prompt into held-out photographs, exports the adapter as safetensors with a manifest, reloads it into a fresh pipeline to verify parity, and finally runs one small optional activity that changes the mask size. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5). No clean-runtime timing has been recorded yet; the downloads alone are about 16.5 GB.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4, then select the Section 3 cell and choose Runtime → Run cell and below: Section 3 builds a fresh, unadapted pipeline (the snapshots are already staged), so the baseline you measure is the frozen model again. Supply your own photographs, masks and captions as a zip holding `captions.csv` beside the files. The table needs the columns `id`, `file` and `caption`, and may carry a `mask` column naming a mask image of the same size (white = repaint, black = keep); a row without a mask gets the deterministic centre mask, and the cell reports how many records used it. Photographs must have a shorter side of at least 256 px and a longer side of at most 4096 px; you need at least four photographs, and at least one caption with three or more photographs so that a held-out record exists. Set `BYOD_PATH` to a zip or directory already in the runtime to skip the upload dialog. Your records are split by caption into training, validation and test sets and flow through the same contract — validation, prompt encoding, frozen baseline, adaptation, held-out evaluation, inference, artifact export and reload parity. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

Kandinsky 2.2 is a two-stage latent diffusion model from ai-forever (Razzhigaev et al., 2023). A diffusion prior with CLIP text and image encoders maps a caption to a CLIP image embedding. The inpainting decoder's UNet (1,253,074,568 parameters) then denoises a 64 × 64 × 4 latent. Its input has nine channels: the four noisy latent channels, the four latent channels of the photograph with the repaint region zeroed, and one **keep mask** channel (1 = keep). A MoVQ autoencoder encodes the photograph to latents and decodes the result to 512 × 512 RGB pixels. Three pinned snapshots make one inpainting system: the decoder (5.28 GB safetensors), the shared diffusion prior (10.57 GB safetensors) and, for evaluation only, a CLIP ViT-B/32 scorer.

Two properties are handled in the open. **The prior pipeline can be released before training**: Section 5 encodes every caption the notebook uses into image embeddings, each prior run seeded from a SHA-256 of its caption, and releases the prior so that the UNet, the MoVQ, the scorer and a training graph fit on a 16 GB GPU. **Inpainting has no single ground truth** for the repainted region, so the notebook reads three kinds of number: the held-out *denoising loss* (the training objective on photographs the model never trained on), the *preservation* of the kept region, and CLIP scores of the outputs between a mean-colour-fill floor and the original-photograph ceiling. None of these is a human judgement of image quality.

**Learning objectives:** by the end of this notebook you will be able to **explain** what the mask, the keep-mask channel, the prior and the latent are, and how an inpainting UNet uses them; **inspect** a validated, digest-pinned photograph dataset and **identify** what validation refuses; **predict**, then **compare**, the frozen and the adapted model on identical held-out inputs; **interpret** a denoising loss, a preservation score and a CLIP similarity, and **explain** why none of them alone measures image quality; **apply** a bounded LoRA fine-tuning with explicit hyperparameters; **verify** that an exported safetensors adapter reloads against the pinned base with the same outputs; and **diagnose** how the mask size changes what the model repaints and preserves.

**This notebook does not demonstrate:** text-to-image generation without a source photograph, depth or ControlNet conditioning, outpainting, full fine-tuning, DreamBooth identifiers, safety filtering of prompts or images, human preference studies, FID, prompt engineering, and any claim that a CLIP score, a preservation score or a denoising loss measures image quality.

## How to use this notebook

**Who this notebook is for.** Learners who can open a hosted notebook, run cells in order and read short Python, and who want to see how a diffusion inpainting model is measured and adapted. No prior diffusion-model experience is assumed: each term is explained where it is first used, and the glossary below collects them. A **GPU** runtime is required (Colab: Runtime → Change runtime type → T4 GPU).

- **Run all** from a fresh GPU runtime (Runtime → Run all). Every cell runs in order without editing; nothing asks for input on the default path, and no manual restart is needed.
- **Infrastructure** cells (Sections 1–3) are collapsed and titled *Infrastructure*. They install pinned packages, carry the repository's code and verify the model files. You may run them without studying their implementation.
- **Form fields** (`# @param`) such as `USE_BYOD`, `STEPS` or `EPOCHS` can be changed in the form view. The defaults are the canonical path; change them only after one complete run.
- Each section is tagged by kind: **[Concept]** explains an idea, **[Evaluation practice]** concerns fair measurement, and **[Engineering]** concerns reproducibility.
- Before each principal result you are asked to **predict**. Write your prediction down first. After the result, a **What to notice** note describes normal output, and a **Check your reasoning** box holds a sample answer: open it only after you have answered.

**Roadmap.** 1–3 set up and verify the model (Infrastructure) → 4 photographs, masks, validation and splits → 5 prompt encoding → 6 the frozen baseline → 7 LoRA fine-tuning → 8 the paired held-out comparison → 9 a new prompt, export and fresh reload → 10 change one thing: the mask size → interpretation, conclusion and troubleshooting. Optional: re-run from Section 4 with your own photographs and masks (Bring Your Own Data).

**Input → Model/System → Output.** A photograph, a mask marking the region to repaint (white = repaint, black = keep) and a caption → prior (caption → CLIP image embedding) → inpainting UNet (denoises a latent, conditioned on the embedding, the kept latents and the keep mask) → MoVQ decoder → a 512 × 512 photograph whose masked region is repainted to match the caption and whose kept region should be preserved.

<details>
<summary><strong>Glossary</strong> — open when a term is unfamiliar</summary>

| Term | Meaning in this notebook |
|---|---|
| **Inpainting** | Repainting a masked region of a photograph so that it matches a caption and blends with the rest. |
| **Mask** | A greyscale image the size of the photograph. In this repository white (255) marks the region to repaint and black (0) the region to keep. |
| **Keep mask** | The inverse mask (1 = keep) that the Kandinsky 2.2 inpainting UNet reads as its ninth input channel; `unet_keep_mask` converts one into the other. |
| **Latent** | A compressed 64 × 64 × 4 representation of a 512 × 512 image, produced by the MoVQ encoder; diffusion runs on latents, not pixels. |
| **Diffusion / denoising** | Generation starts from random noise and removes it step by step; the UNet predicts the noise to remove. |
| **Prior** | A second diffusion model that turns a caption into a CLIP image embedding, the condition the UNet reads. |
| **Classifier-free guidance** | Mixing a conditioned and an unconditioned prediction; `GUIDANCE_SCALE` sets how strongly the caption steers the result. |
| **Denoising loss** | The mean squared error between predicted and true noise: the training objective, measured here on held-out photographs. |
| **Preservation (PSNR, SSIM)** | How closely the kept region of the output matches the input photograph. Higher is closer. |
| **CLIP prompt similarity** | A frozen CLIP model's similarity score between an image and its caption: an automated proxy, not a human judgement. |
| **LoRA** | Low-rank adapter matrices added to the UNet's attention projections; only they are trained, the base model stays frozen. |
| **Held-out** | Photographs never used for training or for choosing the kept epoch (the test split). |
| **Paired comparison** | Measuring the frozen and the adapted model on identical inputs, noise and seeds, so the difference is the adaptation. |

</details>

## Prerequisites

- **Runtime:** a fresh supported **GPU** runtime (Google Colab T4 or better, or a Jupyter kernel with a CUDA GPU of at least 15 GB and Python 3.12). The prior runs in float16 while it encodes captions and is then released; the UNet and MoVQ run in float16; the LoRA tensors are kept in float32 and training uses float16 autocast with loss scaling. CPU-only runtimes are not supported for this notebook. About 20 GB of free disk is needed for the snapshots.
- **Knowledge:** basic Python and notebooks. Helpful but not required: what a diffusion model does at inference, what classifier-free guidance is, and why a training loss is not a quality score. The glossary above covers the terms.
- **Weights:** the decoder, the diffusion prior and the CLIP scorer are all safetensors; nothing is unpickled and no Hub-hosted code is executed — the model classes come from `diffusers`, `transformers` and `peft` on PyPI. The Kandinsky weights are released under Apache-2.0; the scorer is MIT.
- **Data contract:** a record is `{{id, image, caption}}` plus an optional `mask_image` — an RGB photograph with shorter side at least 256 px and longer side at most 4096 px, a caption of 1..1000 characters, and a greyscale mask of the same size (255 = repaint). A record without a mask gets the deterministic centre mask (the middle half of each side). Photograph and mask are resized together so the shorter side is 512 px and centre-cropped to 512 × 512. Validation is structural: nothing checks that a caption describes its photograph or that a mask covers anything sensible.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — photographs of identifiable people, licensed stock images or client material are exactly that. The default path uploads nothing, and uploaded BYOD files are processed only inside this runtime.
- **External access (data):** besides the Hub, the default path fetches 60 pinned photographs (about 6 MB) from the public iNaturalist open-data bucket `inaturalist-open-data.s3.amazonaws.com` over HTTPS, digest-verified before decoding; every photo is CC0 and its observation page is recorded.
- **External access:** the Hugging Face Hub only, to fetch the pinned `kandinsky-community/kandinsky-2-2-decoder-inpaint` snapshot (~16468 MB in total) at revision `db790ad5cbca…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime · [Engineering]

> **Infrastructure.** The code cell(s) in this section are collapsed. You may run them without studying their implementation; they set up a reproducible, verified environment and are not learning objectives.

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `diffusers`, `transformers`, `peft` versions, and whether CUDA is available.

**Expected result:** one dictionary; `'cuda': True` on a GPU runtime. Installation messages before it are normal.

In [ ]:
# @title Infrastructure: install the pinned runtime and print the versions
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'diffusers==0.40.0',
    'transformers==5.17.0',
    'peft==0.21.0',
    'torchao==0.18.0',
    'accelerate==1.15.0',
    'tokenizers==0.23.2',
    'sentencepiece==0.2.2',
    'protobuf==7.36.2',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'kandinsky-inpainting-pipeline',
    'repository_revision': '89afe7e81d3ba9cccc9465a05f9e7f01c8c6d016',
    'embedded_module': 'src/kandinsky_inpainting_pipeline/pipeline.py',
    'embedded_modules': ['src/kandinsky_inpainting_pipeline/metrics.py', 'src/kandinsky_inpainting_pipeline/pipeline.py', 'src/kandinsky_inpainting_pipeline/samples.py'],
    'module_sha256': '2d6c1e6e0f1ed12dbdffae5ba416aba2b6e9d83591b05726a62d4a09f2292cbf',
    'generator': 'build_notebook.py/3',
    'notebook_spec': '2.2',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, diffusers, transformers, peft
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'diffusers': diffusers.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/kandinsky_inpainting_pipeline/` @ `89afe7e81d3b`) · [Engineering]

> **Infrastructure.** The code cell(s) in this section are collapsed. You may run them without studying their implementation; they set up a reproducible, verified environment and are not learning objectives.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/kandinsky_inpainting_pipeline/metrics.py`

In [ ]:
"""Metrics and evaluation helpers for the Kandinsky 2.2 inpainting pipeline.

Provides:
- Unmasked region preservation metrics (PSNR and SSIM on preserved non-inpaint pixels)
- CLIP scorer for prompt-image alignment
"""
from __future__ import annotations

import math
from typing import Any

import numpy as np
from PIL import Image

SCORER_ID = "laion/CLIP-ViT-B-32-laion2B-s34B-b79K"
SCORER_REVISION = "1a25a446712ba5ee05982a381eed697ef9b435cf"


def _projected(features: Any) -> Any:
    """Return the projected embedding across supported Transformers return types."""
    return features if hasattr(features, "shape") else features.pooler_output


def compute_psnr(
    im1: np.ndarray | Image.Image,
    im2: np.ndarray | Image.Image,
    mask: np.ndarray | Image.Image | None = None,
) -> float:
    """Compute Peak Signal-to-Noise Ratio (PSNR) in decibels (dB).

    If `mask` is provided, computation is restricted to pixels where mask is True (preserved region).
    """
    a = np.asarray(im1).astype(np.float32)
    b = np.asarray(im2).astype(np.float32)
    if mask is not None:
        m = np.asarray(mask) > 0
        if m.ndim == 2:
            m = m[:, :, None]
        v1 = a[np.broadcast_to(m, a.shape)]
        v2 = b[np.broadcast_to(m, b.shape)]
        if len(v1) == 0:
            return 100.0
        mse = float(np.mean((v1 - v2) ** 2))
    else:
        mse = float(np.mean((a - b) ** 2))

    if mse < 1e-10:
        return 100.0
    return float(10.0 * math.log10((255.0 ** 2) / mse))


def compute_ssim(
    im1: np.ndarray | Image.Image,
    im2: np.ndarray | Image.Image,
    mask: np.ndarray | Image.Image | None = None,
) -> float:
    """Compute Structural Similarity Index (SSIM) in range [-1, 1].

    If `mask` is provided, computes SSIM over the masked region where mask is True (preserved region).
    """
    a = np.asarray(im1).astype(np.float32)
    b = np.asarray(im2).astype(np.float32)
    c1 = (0.01 * 255.0) ** 2
    c2 = (0.03 * 255.0) ** 2

    if mask is not None:
        m = np.asarray(mask) > 0
        if m.ndim == 2:
            m = m[:, :, None]
        v1 = a[np.broadcast_to(m, a.shape)]
        v2 = b[np.broadcast_to(m, b.shape)]
        if len(v1) == 0:
            return 1.0
        mu1, mu2 = float(np.mean(v1)), float(np.mean(v2))
        s1, s2 = float(np.var(v1)), float(np.var(v2))
        s12 = float(np.mean((v1 - mu1) * (v2 - mu2)))
    else:
        mu1, mu2 = float(np.mean(a)), float(np.mean(b))
        s1, s2 = float(np.var(a)), float(np.var(b))
        s12 = float(np.mean((a - mu1) * (b - mu2)))

    numerator = (2.0 * mu1 * mu2 + c1) * (2.0 * s12 + c2)
    denominator = (mu1 ** 2 + mu2 ** 2 + c1) * (s1 + s2 + c2)
    return float(numerator / (denominator + 1e-10))


def score_inpainting_preservation(
    original_images: list[Image.Image],
    inpainted_images: list[Image.Image],
    mask_images: list[Image.Image],
) -> dict[str, Any]:
    """Score preservation of unmasked regions.

    mask convention: 1 (or 255) indicates region to inpaint; 0 indicates preserved region.
    Preservation metrics evaluate on the region where mask == 0.
    """
    psnr_scores = []
    ssim_scores = []
    for orig, inpaint, mask in zip(original_images, inpainted_images, mask_images, strict=True):
        m_arr = np.asarray(mask)
        # unmasked region is where mask == 0
        unmasked = (m_arr == 0)
        psnr_scores.append(compute_psnr(orig, inpaint, mask=unmasked))
        ssim_scores.append(compute_ssim(orig, inpaint, mask=unmasked))

    return {
        "n_samples": len(original_images),
        "mean_unmasked_psnr_db": float(np.mean(psnr_scores)),
        "mean_unmasked_ssim": float(np.mean(ssim_scores)),
        "psnr_per_sample": psnr_scores,
        "ssim_per_sample": ssim_scores,
    }


class ClipScorer:
    """Frozen CLIP ViT-B/32 scorer for prompt alignment."""

    def __init__(self, device: str = "cpu", weights_dir: str | None = None):
        self.device = device
        self.weights_dir = weights_dir
        self._model = None
        self._processor = None

    def _ensure_loaded(self) -> None:
        if self._model is not None:
            return
        import torch
        from transformers import CLIPModel, CLIPProcessor

        path = self.weights_dir or SCORER_ID
        self._processor = CLIPProcessor.from_pretrained(path)
        self._model = CLIPModel.from_pretrained(path, torch_dtype=torch.float32).to(self.device).eval()

    def score(self, images: list[Image.Image], prompts: list[str]) -> list[float]:
        self._ensure_loaded()
        import torch

        inputs = self._processor(text=prompts, images=images, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            outputs = self._model(**inputs)
            logits_per_image = outputs.logits_per_image  # [N, N]
            similarities = torch.diagonal(logits_per_image).cpu().tolist()
        return similarities


def score_generations(scorer: ClipScorer, images: list[Image.Image], prompts: list[str]) -> dict[str, Any]:
    sims = scorer.score(images, prompts)
    return {
        "mean_prompt_similarity": float(np.mean(sims)),
        "prompt_similarities": sims,
    }

**Module 2/3:** `src/kandinsky_inpainting_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""DIMER-oriented pipeline for Kandinsky 2.2 inpainting (decoder inpaint + shared prior).

Exposes:
- Snapshot verification and staging at immutable revisions
- Inpaint UNet construction with rank-8 LoRA adapter injection
- Lazy prior pipeline execution with prompt-embedding cache
- Inpainting generation with unmasked preservation guarantees
- Held-out evaluation reporting denoising MSE, PSNR and SSIM on unmasked regions
- Bounded LoRA fine-tuning and portable safetensors artifact serialization
"""
from __future__ import annotations

import gc
import hashlib
import json
import math
import shutil
import time
import warnings
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .metrics import compute_psnr, compute_ssim` removed — names are kernel globals defined by the carried modules

MODEL_ID = "kandinsky-community/kandinsky-2-2-decoder-inpaint"
MODEL_REVISION = "db790ad5cbcabed886f069ef2710774657621702"
MODEL_KEY = "kandinsky-2-2-decoder-inpaint"

PRIOR_ID = "kandinsky-community/kandinsky-2-2-prior"
PRIOR_REVISION = "9fc51ad5732afc5d031724219d22e6c42179c5a8"
PRIOR_KEY = "kandinsky-2-2-prior"

SCORER_ID = "laion/CLIP-ViT-B-32-laion2B-s34B-b79K"
SCORER_REVISION = "1a25a446712ba5ee05982a381eed697ef9b435cf"
SCORER_KEY = "clip-vit-b-32-laion2b"

MODEL_LICENSE = "apache-2.0"
ARTIFACT_FORMAT = "org.valcorza.kandinsky-inpainting.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
MANIFEST_NAME = "dimer-base-manifest.json"

_WEIGHTS_ROOT = Path.cwd() / "weights"  # standalone rewrite (build_notebook.py): working-directory-relative
DEFAULT_WEIGHTS_DIR = _WEIGHTS_ROOT / MODEL_KEY
DEFAULT_PRIOR_DIR = _WEIGHTS_ROOT / PRIOR_KEY
DEFAULT_SCORER_DIR = _WEIGHTS_ROOT / SCORER_KEY

UNET_PARAMETERS = 1253074568
UNET_TENSORS = 724
LORA_RANK = 8
LORA_ALPHA = 8
LORA_TARGETS = ("to_q", "to_k", "to_v", "to_out.0")
LORA_TENSORS = 176
LORA_PARAMETERS = 1646592

DEFAULT_STEPS = 20
DEFAULT_GUIDANCE = 4.0
MAX_STEPS = 100
MAX_GUIDANCE = 20.0
RESOLUTION = 512
LATENT_CHANNELS = 4
MAX_CAPTION_CHARS = 1_000
MIN_IMAGE_SIDE = 256
MAX_IMAGE_SIDE = 4_096
MIN_TRAIN_RECORDS = 4
MAX_RECORDS = 2_000
NUM_TRAIN_TIMESTEPS = 1_000
EVAL_TIMESTEPS = (100, 300, 500, 700, 900)
NEGATIVE_PROMPT = ""


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def prompt_seed(prompt: str) -> int:
    """Seed for the prior's sampler: the first 4 bytes of SHA-256(prompt), masked to 31 bits.

    Stable across processes, unlike ``hash(prompt)``, which Python salts per interpreter.
    """
    return int.from_bytes(hashlib.sha256(prompt.encode("utf-8")).digest()[:4], "big") & 0x7FFFFFFF


def _read_manifest(dir_path: Path) -> dict[str, Any]:
    manifest_path = dir_path / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    return json.loads(manifest_path.read_text(encoding="utf-8"))


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest = _read_manifest(root)
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest.get("files", []):
        relative_path = entry["path"]
        target = root / relative_path
        if not target.is_file():
            raise FileNotFoundError(f"snapshot file missing: {target}")
        if target.stat().st_size != entry["bytes"]:
            raise ValueError(f"{relative_path}: size {target.stat().st_size} != manifest {entry['bytes']}")
        digest = _sha256_file(target)
        if digest != entry["sha256"]:
            raise ValueError(f"{relative_path}: sha256 {digest} != manifest {entry['sha256']}")
        if not relative_path.endswith((".safetensors", ".json", ".model", ".txt", ".md")):
            raise ValueError(f"{relative_path}: unexpected file type in a code-free snapshot")
    return manifest


def verify_snapshot(weights_dir: str | Path = DEFAULT_WEIGHTS_DIR) -> dict[str, Any]:
    """Verify that all files in the decoder inpaint snapshot exist and match recorded hashes."""
    return _verify_manifest(Path(weights_dir), MODEL_ID, MODEL_REVISION)


def verify_prior_snapshot(prior_dir: str | Path = DEFAULT_PRIOR_DIR) -> dict[str, Any]:
    """Verify that all files in the prior snapshot exist and match recorded hashes."""
    return _verify_manifest(Path(prior_dir), PRIOR_ID, PRIOR_REVISION)


def verify_scorer_snapshot(scorer_dir: str | Path = DEFAULT_SCORER_DIR) -> dict[str, Any]:
    """Verify that all files in the CLIP scorer snapshot exist and match recorded hashes."""
    return _verify_manifest(Path(scorer_dir), SCORER_ID, SCORER_REVISION)


def _hub_download(relative_path: str, root: Path, model_id: str, revision: str) -> None:
    from huggingface_hub import hf_hub_download

    cached = hf_hub_download(model_id, relative_path, revision=revision)
    target = root / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached, target)


def _stage_missing(
    root: Path,
    model_id: str,
    revision: str,
    allow_download: bool,
    downloader: Callable[[str, Path], None] | None,
) -> list[str]:
    manifest = _read_manifest(root)
    if manifest.get("modelId") != model_id or manifest.get("revision") != revision:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {model_id}@{revision}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest.get("files", []) if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them")
    fetch = downloader or (lambda relative_path, destination: _hub_download(relative_path, destination, model_id, revision))
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def stage_missing_files(
    weights_dir: str | Path = DEFAULT_WEIGHTS_DIR,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Stage missing files for the inpaint decoder snapshot."""
    return _stage_missing(Path(weights_dir), MODEL_ID, MODEL_REVISION, allow_download, downloader)


def stage_missing_prior_files(
    prior_dir: str | Path = DEFAULT_PRIOR_DIR,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Stage missing files for the prior snapshot."""
    return _stage_missing(Path(prior_dir), PRIOR_ID, PRIOR_REVISION, allow_download, downloader)


def stage_missing_scorer_files(
    scorer_dir: str | Path = DEFAULT_SCORER_DIR,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Stage missing files for the CLIP scorer snapshot."""
    return _stage_missing(Path(scorer_dir), SCORER_ID, SCORER_REVISION, allow_download, downloader)


INPUT_SCHEMA: dict[str, Any] = {
    "record": "{id, image, mask_image?, caption}: an RGB image, optional inpainting mask, and prompt",
    "image_side": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "resolution": RESOLUTION,
    "preprocessing": (
        f"image and mask are resized together so the shorter side is {RESOLUTION} px and centre-cropped "
        f"to {RESOLUTION}x{RESOLUTION}"
    ),
    "caption_chars": [1, MAX_CAPTION_CHARS],
    "records": [MIN_TRAIN_RECORDS, MAX_RECORDS],
    "generation": {"steps": [1, MAX_STEPS], "guidance_scale": [1.0, MAX_GUIDANCE], "size": RESOLUTION},
    "validation": (
        "record shape, image decodability and side limits, optional mask shape, caption length and duplicate ids only. "
        "Nothing checks semantic alignment: any RGB image with any string is accepted"
    ),
}


def _load_image(value: Any, *, mode: str, label: str) -> Any:
    from PIL import Image

    image = value
    if isinstance(value, str | Path):
        path = Path(value)
        if not path.is_file():
            raise ValueError(f"{label} file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label} must be a PIL.Image.Image or a file path")
    return image.convert(mode)


def create_center_mask(image_or_size: Any) -> Any:
    """Create a binary center-box mask covering the middle half of an image."""
    from PIL import Image, ImageDraw

    size = image_or_size if isinstance(image_or_size, tuple) else image_or_size.size
    width, height = size
    mask = Image.new("L", (width, height), 0)
    ImageDraw.Draw(mask).rectangle((width // 4, height // 4, 3 * width // 4, 3 * height // 4), fill=255)
    return mask


def unet_keep_mask(inpaint_mask: Any) -> Any:
    """Convert an inpaint mask (1 = region to repaint, the repository convention) into the Kandinsky 2.2 inpainting
    UNet's mask channel (1 = region to keep).

    The repository convention (255 / 1 = repaint) is what records, ``create_center_mask`` and ``generate`` accept.
    The UNet expects the opposite convention: diffusers v0.40.0
    ``pipelines/kandinsky2_2/pipeline_kandinsky2_2_inpainting.py`` inverts the user mask (``mask = 1 - mask``,
    L238), multiplies the image latents by it (L452) and concatenates it as the last UNet input channel (L479).
    Works on NumPy arrays and torch tensors alike.
    """
    return 1.0 - inpaint_mask


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/image/caption")
    for key in ("id", "image", "caption"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    record_id = record["id"]
    if not isinstance(record_id, str) or not record_id or len(record_id) > 64:
        raise ValueError(f"{label}: id must be a non-empty string of at most 64 characters")
    image = _load_image(record["image"], mode="RGB", label=f"{label}: image")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"{label}: image sides must be within {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px, got {image.size}")
    caption = record["caption"]
    if not isinstance(caption, str) or not caption.strip() or len(caption) > MAX_CAPTION_CHARS:
        raise ValueError(f"{label}: caption must be a non-empty string of at most {MAX_CAPTION_CHARS} characters")
    mask = (
        _load_image(record["mask_image"], mode="L", label=f"{label}: mask_image")
        if "mask_image" in record
        else create_center_mask(image)
    )
    if mask.size != image.size:
        raise ValueError(f"{label}: mask_image size {mask.size} must match image size {image.size}")
    item = {"id": record_id, "image": image, "mask_image": mask, "caption": caption.strip()}
    for key in ("label", "common_name", "scientific_name", "observer", "inat_photo_id", "inat_observation_url", "source_id"):
        if key in record:
            item[key] = record[key]
    return item


def image_digest(image: Any) -> str:
    """Return a stable SHA-256 over decoded RGB pixels and dimensions."""
    rgb = image.convert("RGB")
    return hashlib.sha256(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes()).hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = []
    for record in records:
        mask = record.get("mask_image")
        mask_digest = hashlib.sha256(mask.convert("L").tobytes()).hexdigest() if mask is not None else None
        payload.append([record["id"], image_digest(record["image"]), mask_digest, record["caption"]])
    encoded = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_TRAIN_RECORDS,
    max_records: int = MAX_RECORDS,
) -> dict[str, Any]:
    """Validate inpainting records before importing any model dependency."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, caption} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    crops = 0
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        crops += item["image"].size[0] != item["image"].size[1]
        checked.append(item)
    sides = [min(item["image"].size) for item in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "n_captions": len({item["caption"] for item in checked}),
        "shorter_side": {"min": min(sides), "max": max(sides)},
        "centre_cropped": crops,
        "resolution": RESOLUTION,
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def validate_inputs(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record and report its deterministic preprocessing shape."""
    item = _check_record(record, 0)
    width, height = item["image"].size
    scale = RESOLUTION / min(width, height)
    return {
        "id": item["id"],
        "size": (width, height),
        "resized_to": (round(width * scale), round(height * scale)),
        "centre_crop": (RESOLUTION, RESOLUTION),
        "caption_chars": len(item["caption"]),
    }


def validate_prompts(prompts: Sequence[str]) -> list[str]:
    """Validate and normalize generation prompts."""
    if isinstance(prompts, str) or not isinstance(prompts, Sequence) or not prompts:
        raise ValueError("prompts must be a non-empty list of strings")
    normalized = []
    for index, prompt in enumerate(prompts):
        if not isinstance(prompt, str) or not prompt.strip() or len(prompt) > MAX_CAPTION_CHARS:
            raise ValueError(f"prompts[{index}] must be a non-empty string of at most {MAX_CAPTION_CHARS} characters")
        normalized.append(prompt.strip())
    return normalized


def _resize_and_crop(image: Any, *, target_size: int, resample: Any) -> Any:
    width, height = image.size
    scale = target_size / min(width, height)
    resized_size = (max(target_size, round(width * scale)), max(target_size, round(height * scale)))
    resized = image.resize(resized_size, resample)
    left = (resized_size[0] - target_size) // 2
    top = (resized_size[1] - target_size) // 2
    return resized.crop((left, top, left + target_size, top + target_size))


def preprocess_image(image: Any, *, target_size: int = RESOLUTION) -> Any:
    """Resize the shorter side and center-crop an image to a square RGB image."""
    from PIL import Image

    return _resize_and_crop(image.convert("RGB"), target_size=target_size, resample=Image.Resampling.BICUBIC)


def preprocess_image_and_mask(image: Any, mask_image: Any, *, target_size: int = RESOLUTION) -> tuple[Any, Any]:
    """Apply identical resize/crop geometry to an RGB image and its binary mask."""
    from PIL import Image

    rgb = image.convert("RGB")
    mask = mask_image.convert("L")
    if rgb.size != mask.size:
        raise ValueError(f"mask_image size {mask.size} must match image size {rgb.size}")
    return (
        _resize_and_crop(rgb, target_size=target_size, resample=Image.Resampling.BICUBIC),
        _resize_and_crop(mask, target_size=target_size, resample=Image.Resampling.NEAREST),
    )


def synthetic_image(*, width: int = 640, height: int = 480, seed: int = 0) -> Any:
    """Create a deterministic RGB image for examples and smoke tests."""
    from PIL import Image

    rng = np.random.default_rng(seed)
    array = rng.integers(0, 256, size=(height, width, 3), dtype=np.uint8)
    return Image.fromarray(array, mode="RGB")


def synthetic_records(n: int = 8, *, seed: int = 0) -> list[dict[str, Any]]:
    """Create deterministic inpainting records for local examples."""
    records = []
    for index in range(n):
        image = synthetic_image(seed=seed + index)
        records.append(
            {
                "id": f"rec-{index:03d}",
                "image": image,
                "mask_image": create_center_mask(image),
                "caption": "a photo of a red bird" if index % 2 == 0 else "a photo of a blue bird",
            }
        )
    return records


def _lora_config() -> Any:
    from peft import LoraConfig

    return LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, init_lora_weights="gaussian", target_modules=list(LORA_TARGETS))


def lora_parameter_names(unet: Any) -> list[str]:
    """Return sorted list of attached LoRA parameter names."""
    return sorted(name for name, _p in unet.named_parameters() if ".lora_A." in name or ".lora_B." in name)


def build_unet(weights_dir: Path, *, dtype: Any, use_lora: bool) -> Any:
    """Build UNet2DConditionModel from weights_dir, optionally attaching LoRA."""
    from diffusers import UNet2DConditionModel

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = UNet2DConditionModel.from_pretrained(str(weights_dir), subfolder="unet", torch_dtype=dtype)

    n_params = sum(p.numel() for p in model.parameters())
    if n_params != UNET_PARAMETERS or len(model.state_dict()) != UNET_TENSORS:
        raise ValueError(
            f"unet parameter mismatch: got {n_params} parameters in {len(model.state_dict())} tensors, "
            f"expected {UNET_PARAMETERS} / {UNET_TENSORS}"
        )
    for p in model.parameters():
        p.requires_grad_(False)

    if use_lora:
        from peft import inject_adapter_in_model

        inject_adapter_in_model(_lora_config(), model, adapter_name="default")
        names = lora_parameter_names(model)
        if len(names) != LORA_TENSORS:
            raise ValueError(f"adapter attached {len(names)} LoRA tensors, expected {LORA_TENSORS}")
        import torch

        name_set = set(names)
        for name, param in model.named_parameters():
            if name in name_set:
                # LoRA tensors are kept in float32 so AdamW state and updates do not underflow in float16.
                param.data = param.data.to(torch.float32)
                param.requires_grad_(True)
                if ".lora_B." in name:
                    param.data.zero_()
            else:
                param.requires_grad_(False)
    return model


class KandinskyInpaintPipeline:
    """DIMER pipeline for Kandinsky 2.2 Inpainting with optional LoRA adaptation."""

    def __init__(
        self,
        unet: Any,
        movq: Any,
        scheduler_config: dict[str, Any],
        *,
        device: str,
        dtype: Any,
        prior_dir: Path,
        weights_dir: Path,
        source: str = "local-snapshot (verified safetensors)",
        use_lora: bool = False,
    ):
        self.unet = unet
        self.movq = movq
        self.scheduler_config = scheduler_config
        self.device = device
        self.dtype = dtype
        self.prior_dir = prior_dir
        self.weights_dir = weights_dir
        self.source = source
        self.use_lora = use_lora
        self.adapter: dict[str, Any] | None = None
        self._prior_pipeline = None
        self._prompt_cache: dict[str, dict[str, Any]] = {}

    @classmethod
    def from_pretrained(
        cls,
        weights_dir: str | Path = DEFAULT_WEIGHTS_DIR,
        prior_dir: str | Path = DEFAULT_PRIOR_DIR,
        *,
        device: str | None = None,
        use_lora: bool = False,
        allow_download: bool = False,
    ) -> KandinskyInpaintPipeline:
        """Instantiate pipeline from verified snapshot directories."""
        import torch
        from diffusers import VQModel

        w_path = Path(weights_dir)
        p_path = Path(prior_dir)

        if allow_download:
            stage_missing_files(w_path, allow_download=True)
            stage_missing_prior_files(p_path, allow_download=True)

        verify_snapshot(w_path)
        verify_prior_snapshot(p_path)

        dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if dev == "cuda" else torch.float32

        unet = build_unet(w_path, dtype=dtype, use_lora=use_lora).to(dev)
        movq = VQModel.from_pretrained(str(w_path), subfolder="movq", torch_dtype=dtype).to(dev).eval()
        for p in movq.parameters():
            p.requires_grad_(False)

        sched_path = w_path / "scheduler" / "scheduler_config.json"
        scheduler_config = json.loads(sched_path.read_text(encoding="utf-8"))

        return cls(
            unet=unet,
            movq=movq,
            scheduler_config=scheduler_config,
            device=dev,
            dtype=dtype,
            prior_dir=p_path,
            weights_dir=w_path,
            use_lora=use_lora,
        )

    def _get_prior(self) -> Any:
        if self._prior_pipeline is not None:
            return self._prior_pipeline
        import warnings

        from diffusers import KandinskyV22PriorPipeline

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self._prior_pipeline = KandinskyV22PriorPipeline.from_pretrained(
                str(self.prior_dir),
                torch_dtype=self.dtype,
            ).to(self.device)
        return self._prior_pipeline

    def encode_prompts(self, prompts: Sequence[str]) -> dict[str, Any]:
        """Encode prompts with the prior into CLIP image embeddings.

        Each prompt's prior sampler is seeded with ``prompt_seed(prompt)``, so a prompt's embedding does not depend
        on the other prompts encoded with it or on Python's per-process string-hash salt.
        """
        distinct = sorted(set([NEGATIVE_PROMPT, *validate_prompts(prompts)]))
        missing = [p for p in distinct if p not in self._prompt_cache]
        if not missing:
            return {"n_cached": len(self._prompt_cache), "n_new": 0}
        import torch

        prior = self._get_prior()
        prior.set_progress_bar_config(disable=True)
        for prompt in missing:
            generator = torch.Generator(device=self.device).manual_seed(prompt_seed(prompt))
            out = prior(prompt=prompt, num_inference_steps=25, generator=generator)
            self._prompt_cache[prompt] = {
                "image_embeds": out.image_embeds[0].detach().to("cpu", self.dtype),
                "negative_image_embeds": out.negative_image_embeds[0].detach().to("cpu", self.dtype),
            }
        return {"n_cached": len(self._prompt_cache), "n_new": len(missing)}

    def release_prior(self) -> bool:
        """Release the prior pipeline to reclaim GPU VRAM."""
        import torch

        if self._prior_pipeline is None:
            return False
        del self._prior_pipeline
        self._prior_pipeline = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return True

    def export_prompt_cache(self) -> dict[str, Any]:
        return {
            prompt: {
                "image_embeds": entry["image_embeds"].clone(),
                "negative_image_embeds": entry["negative_image_embeds"].clone(),
            }
            for prompt, entry in self._prompt_cache.items()
        }

    def import_prompt_cache(self, cache: Mapping[str, Mapping[str, Any]]) -> int:
        for prompt, entry in cache.items():
            if "image_embeds" not in entry or "negative_image_embeds" not in entry:
                raise ValueError(f"prompt cache entry for {prompt[:40]!r} missing required embedding keys")
            self._prompt_cache[prompt] = {
                "image_embeds": entry["image_embeds"],
                "negative_image_embeds": entry["negative_image_embeds"],
            }
        return len(self._prompt_cache)

    def _embeds(self, prompt: str) -> tuple[Any, Any]:
        if prompt not in self._prompt_cache:
            raise ValueError(f"prompt not encoded; call encode_prompts([...]) first: {prompt[:60]!r}")
        entry = self._prompt_cache[prompt]
        return (
            entry["image_embeds"].to(self.device, self.dtype),
            entry["negative_image_embeds"].to(self.device, self.dtype),
        )

    def _diffusers_pipeline(self) -> Any:
        from diffusers import DDPMScheduler
        from diffusers import KandinskyV22InpaintPipeline as _Upstream

        return _Upstream(
            unet=self.unet,
            scheduler=DDPMScheduler.from_config(self.scheduler_config),
            movq=self.movq,
        )

    def generate(
        self,
        records: Sequence[dict[str, Any]],
        *,
        seed: int = 0,
        steps: int = DEFAULT_STEPS,
        guidance_scale: float = DEFAULT_GUIDANCE,
    ) -> dict[str, Any]:
        """Run inpainting on `{image, mask_image, caption}` records."""
        import torch

        prompts = [r["caption"] for r in records]
        if any(p not in self._prompt_cache for p in [NEGATIVE_PROMPT, *prompts]):
            self.encode_prompts(prompts)

        pipe = self._diffusers_pipeline()
        pipe.set_progress_bar_config(disable=True)
        started = time.perf_counter()
        results = []

        for idx, rec in enumerate(records):
            img, mask = preprocess_image_and_mask(rec["image"], rec["mask_image"], target_size=RESOLUTION)
            prompt = rec["caption"]
            img_emb, neg_emb = self._embeds(prompt)
            generator = torch.Generator(device=self.device).manual_seed(seed + idx)

            with torch.inference_mode():
                out = pipe(
                    prompt=prompt,
                    image=img,
                    mask_image=mask,
                    image_embeds=img_emb[None],
                    negative_image_embeds=neg_emb[None],
                    height=RESOLUTION,
                    width=RESOLUTION,
                    num_inference_steps=steps,
                    guidance_scale=float(guidance_scale),
                    generator=generator,
                    output_type="pil",
                )
            gen_img = out.images[0]
            unmasked = (np.asarray(mask) == 0)
            psnr = compute_psnr(img, gen_img, mask=unmasked)
            ssim = compute_ssim(img, gen_img, mask=unmasked)

            results.append({
                "id": rec.get("id", f"sample-{idx}"),
                "prompt": prompt,
                "image": gen_img,
                "unmasked_psnr_db": round(psnr, 2),
                "unmasked_ssim": round(ssim, 4),
            })

        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "steps": steps,
            "guidance_scale": float(guidance_scale),
            "size": (RESOLUTION, RESOLUTION),
            "results": results,
            "seconds": round(time.perf_counter() - started, 2),
        }

    def _conditioning(self, records: Sequence[Mapping[str, Any]]) -> tuple[Any, Any, Any]:
        """Encode images and masks into latents, masked latents and the latent-resolution inpaint mask.

        Returns ``(latents, masked_latents, inpaint_mask_latents)``: ``masked_latents`` keeps the latents outside the
        repaint region (zero inside it), and ``inpaint_mask_latents`` is 1 inside the repaint region (repository
        convention). ``_predict_noise`` converts it to the UNet's keep-mask channel with ``unet_keep_mask``.
        """
        import torch
        import torch.nn.functional as F

        images = []
        masks = []
        for record in records:
            image, mask = preprocess_image_and_mask(record["image"], record["mask_image"], target_size=RESOLUTION)
            images.append(np.asarray(image, dtype=np.float32) / 127.5 - 1.0)
            masks.append(np.asarray(mask, dtype=np.float32) / 255.0)
        pixels = torch.from_numpy(np.stack(images)).permute(0, 3, 1, 2).to(self.device, self.dtype)
        mask_pixels = torch.from_numpy(np.stack(masks))[:, None].to(self.device, self.dtype)
        with torch.no_grad():
            encoded = self.movq.encode(pixels)
            latents = encoded.latents if hasattr(encoded, "latents") else encoded["latents"]
        mask_latents = F.interpolate(mask_pixels, size=latents.shape[-2:], mode="nearest")
        masked_latents = latents * (1.0 - mask_latents)
        return latents.to(self.dtype), masked_latents.to(self.dtype), mask_latents.to(self.dtype)

    def _noise_scheduler(self) -> Any:
        from diffusers import DDPMScheduler

        return DDPMScheduler.from_config(self.scheduler_config)

    def _predict_noise(
        self,
        noisy: Any,
        timesteps: Any,
        image_embeds: Any,
        masked_latents: Any,
        inpaint_mask_latents: Any,
    ) -> Any:
        """Predict noise from the nine-channel input ``[noisy latents, masked latents, keep mask]``: the channel order
        and keep-mask convention of diffusers v0.40.0 ``KandinskyV22InpaintPipeline`` (L452, L479)."""
        import torch

        keep = unet_keep_mask(inpaint_mask_latents).to(noisy.dtype)
        model_input = torch.cat([noisy, masked_latents.to(noisy.dtype), keep], dim=1)
        output = self.unet(
            sample=model_input,
            timestep=timesteps,
            encoder_hidden_states=None,
            added_cond_kwargs={"image_embeds": image_embeds},
            return_dict=False,
        )
        prediction = output[0] if isinstance(output, tuple) else output.sample
        return prediction[:, :LATENT_CHANNELS]

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        seed: int = 0,
        batch_size: int = 4,
    ) -> dict[str, Any]:
        """Evaluate seeded, paired denoising MSE across the fixed timestep schedule."""
        import torch

        checked = validate_dataset(records, min_records=1)["records"]
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        self.encode_prompts([record["caption"] for record in checked])
        scheduler = self._noise_scheduler()
        per_timestep: dict[int, list[float]] = {timestep: [] for timestep in EVAL_TIMESTEPS}
        per_record: dict[str, float] = {}
        self.unet.eval()
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            latents, masked_latents, mask_latents = self._conditioning(batch)
            image_embeds = torch.stack([self._embeds(record["caption"])[0] for record in batch])
            record_losses = [0.0] * len(batch)
            for timestep in EVAL_TIMESTEPS:
                generator = torch.Generator(device="cpu").manual_seed(seed * 1_000 + timestep + start)
                noise = torch.randn(latents.shape, generator=generator).to(self.device, self.dtype)
                timesteps = torch.full((len(batch),), timestep, device=self.device, dtype=torch.long)
                noisy = scheduler.add_noise(latents.float(), noise.float(), timesteps).to(self.dtype)
                use_amp = self.dtype == torch.float16
                autocast = torch.autocast(device_type=self.device.split(":")[0], dtype=torch.float16, enabled=use_amp)
                with torch.inference_mode(), autocast:
                    prediction = self._predict_noise(noisy, timesteps, image_embeds, masked_latents, mask_latents)
                losses = ((prediction.float() - noise.float()) ** 2).mean(dim=(1, 2, 3))
                for index, value in enumerate(losses.tolist()):
                    per_timestep[timestep].append(value)
                    record_losses[index] += value / len(EVAL_TIMESTEPS)
            for record, value in zip(batch, record_losses, strict=True):
                per_record[record["id"]] = round(value, 6)
        by_timestep = {
            str(timestep): round(sum(values) / len(values), 6) for timestep, values in per_timestep.items()
        }
        return {
            "metric": "denoising_mse",
            "n_records": len(checked),
            "timesteps": list(EVAL_TIMESTEPS),
            "seed": seed,
            "denoising_mse": round(sum(per_record.values()) / len(per_record), 6),
            "by_timestep": by_timestep,
            "per_record": per_record,
            "adapted": self.adapter is not None,
        }

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 4,
        lr: float = 1e-4,
        batch_size: int = 1,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Train only the bounded LoRA tensor set on the inpainting noise-prediction objective.

        AdamW on the float32 LoRA tensors, float16 autocast with a ``GradScaler`` on CUDA (float32 on CPU), loss in
        float32, gradient-norm clipping at 1.0. Epoch 0 records the frozen model; the lowest validation loss is kept.
        """
        if not self.use_lora:
            raise ValueError("adapt() needs a pipeline built with use_lora=True")
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not 0.0 < lr <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 8:
            raise ValueError("batch_size must be an int in 1..8")
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None

        import torch

        prompts = [record["caption"] for record in train_checked]
        if val_checked:
            prompts.extend(record["caption"] for record in val_checked)
        self.encode_prompts(prompts)
        torch.manual_seed(seed)
        started = time.perf_counter()
        model = self.unet
        names = lora_parameter_names(model)
        name_set = set(names)
        for name, parameter in model.named_parameters():
            parameter.requires_grad_(name in name_set)
        parameters = [parameter for name, parameter in model.named_parameters() if name in name_set]
        n_trainable = sum(parameter.numel() for parameter in parameters)
        if n_trainable != LORA_PARAMETERS:
            raise ValueError(f"{n_trainable} trainable parameters, expected {LORA_PARAMETERS}")
        initial_state = {key: value.detach().clone() for key, value in model.state_dict().items() if key in name_set}
        optimizer = torch.optim.AdamW(parameters, lr=lr, weight_decay=0.0)
        use_amp = self.dtype == torch.float16
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        scheduler = self._noise_scheduler()
        generator = torch.Generator(device="cpu").manual_seed(seed)

        conditioning_by_id = {}
        for start in range(0, len(train_checked), 4):
            batch = train_checked[start : start + 4]
            latents, masked_latents, mask_latents = self._conditioning(batch)
            for index, record in enumerate(batch):
                conditioning_by_id[record["id"]] = (
                    latents[index],
                    masked_latents[index],
                    mask_latents[index],
                )

        try:
            history: list[dict[str, Any]] = []
            baseline: dict[str, Any] = {
                "epoch": 0,
                "train_loss": None,
                "note": "frozen model (LoRA at initialization: B = 0)",
                "val_loss": self.evaluate(val_checked, seed=seed)["denoising_mse"] if val_checked else None,
            }
            history.append(baseline)
            if progress:
                progress(baseline)
            best_val = baseline["val_loss"] if baseline["val_loss"] is not None else math.inf
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items() if key in name_set}
            best_epoch = 0
            n_steps = 0
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[index] for index in order[start : start + batch_size]]
                    latents = torch.stack([conditioning_by_id[record["id"]][0] for record in batch])
                    masked_latents = torch.stack([conditioning_by_id[record["id"]][1] for record in batch])
                    mask_latents = torch.stack([conditioning_by_id[record["id"]][2] for record in batch])
                    image_embeds = torch.stack([self._embeds(record["caption"])[0] for record in batch])
                    noise = torch.randn(latents.shape, generator=generator).to(self.device, self.dtype)
                    timesteps = torch.randint(0, NUM_TRAIN_TIMESTEPS, (len(batch),), generator=generator).to(self.device)
                    noisy = scheduler.add_noise(latents.float(), noise.float(), timesteps).to(self.dtype)
                    with torch.autocast(device_type=self.device.split(":")[0], dtype=torch.float16, enabled=use_amp):
                        prediction = self._predict_noise(noisy, timesteps, image_embeds, masked_latents, mask_latents)
                    loss = torch.nn.functional.mse_loss(prediction.float(), noise.float())
                    optimizer.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(parameters, 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    losses.append(float(loss.detach()))
                    n_steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses)}
                entry["val_loss"] = self.evaluate(val_checked, seed=seed)["denoising_mse"] if val_checked else None
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = {key: value.detach().clone() for key, value in model.state_dict().items() if key in name_set}
                    best_epoch = epoch
        except BaseException:
            restored = dict(model.state_dict())
            restored.update(initial_state)
            model.load_state_dict(restored, strict=True)
            model.eval()
            for parameter in model.parameters():
                parameter.requires_grad_(False)
            self.adapter = None
            raise

        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        self.adapter = {
            "method": "LoRA (peft)",
            "rank": LORA_RANK,
            "alpha": LORA_ALPHA,
            "targets": list(LORA_TARGETS),
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(parameter.numel() for parameter in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "optimizer": "AdamW (weight_decay 0, grad-norm clip 1.0)",
            "precision": "float16 autocast + GradScaler" if use_amp else "float32",
        }
        return {
            "history": history,
            "best_epoch": best_epoch,
            "best_val_loss": best_val if best_val != math.inf else None,
            "adapter": self.adapter,
            "steps": n_steps,
            "seconds": round(time.perf_counter() - started, 2),
        }

    def save_artifact(self, output_dir: str | Path, *, metadata: Mapping[str, Any] | None = None) -> dict[str, Any]:
        """Save the trained LoRA tensor set as safetensors plus a digest-bound manifest."""
        import safetensors.torch

        if self.adapter is None:
            raise ValueError("no adapter attached; call adapt() or load_adapter() first")
        output = Path(output_dir)
        output.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {key: value.detach().cpu() for key, value in self.unet.state_dict().items() if key in names}
        weights_file = output / ARTIFACT_WEIGHTS_NAME
        safetensors.torch.save_file(tensors, weights_file)
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY},
            "adapter": self.adapter,
            "weights": {
                "file": ARTIFACT_WEIGHTS_NAME,
                "bytes": weights_file.stat().st_size,
                "sha256": _sha256_file(weights_file),
                "n_tensors": len(tensors),
                "n_parameters": sum(tensor.numel() for tensor in tensors.values()),
            },
            "metadata": dict(metadata) if metadata else {},
            "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        (output / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return manifest

    save_adapter = save_artifact

    def load_adapter(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Load and verify an adapter produced by `save_artifact`."""
        import safetensors.torch

        if not self.use_lora:
            raise ValueError("load_adapter() needs a pipeline built with use_lora=True")
        source = Path(artifact_dir)
        manifest_file = source / ARTIFACT_MANIFEST_NAME
        weights_file = source / ARTIFACT_WEIGHTS_NAME
        if not manifest_file.is_file() or not weights_file.is_file():
            raise FileNotFoundError(f"not an adapter directory: {source}")
        manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"unexpected artifact format: {manifest.get('format')!r}")
        base_model = manifest.get("base_model", {})
        if base_model.get("id") != MODEL_ID or base_model.get("revision") != MODEL_REVISION:
            raise ValueError(f"adapter base model {base_model} does not match {MODEL_ID}@{MODEL_REVISION}")
        digest = _sha256_file(weights_file)
        if digest != manifest["weights"]["sha256"]:
            raise ValueError(f"weights digest mismatch: {digest} != manifest {manifest['weights']['sha256']}")
        tensors = safetensors.torch.load_file(weights_file)
        current = dict(self.unet.state_dict())
        current.update({key: value.to(self.device, current[key].dtype) for key, value in tensors.items()})
        self.unet.load_state_dict(current, strict=True)
        self.adapter = manifest["adapter"]
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        weights_dir: str | Path = DEFAULT_WEIGHTS_DIR,
        prior_dir: str | Path = DEFAULT_PRIOR_DIR,
        device: str | None = None,
    ) -> KandinskyInpaintPipeline:
        """Instantiate a verified pipeline and load an adapter artifact."""
        pipe = cls.from_pretrained(weights_dir=weights_dir, prior_dir=prior_dir, device=device, use_lora=True)
        pipe.load_adapter(artifact_dir)
        return pipe

**Module 3/3:** `src/kandinsky_inpainting_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Captioned-image dataset contract for adapting the generator: the pinned iNaturalist bird sample, seeded
splitting, generation prompts, BYOD loaders and CSV export.

The default dataset is **real** and narrow on purpose: 60 CC0-licensed, research-grade iNaturalist photographs of
six common North American birds (10 per species, one per observer per species), a subset of the corpus the
fleet's DINOv2 and ViT rows pinned on 2026-09-19, pinned here by photo id, byte size and SHA-256 of the served
`medium` JPEG (about 500 px on the longer side). Every file is fetched from the iNaturalist open-data bucket at
run time and refused on any byte-size or SHA-256 mismatch; the repository redistributes none of the photographs.
Each record keeps the observation id and observer login so every image is traceable to its public observation
page. Captions are generated from the species names by one template, so the adaptation teaches the generator
what these six names look like in this kind of photograph.

A record is ``{id, image, caption}`` plus an optional ``mask_image`` (255 = region to repaint): a PIL image (or a path
to one), the caption that describes it, and the mask; a record without a mask gets the deterministic centre mask.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MIN_TRAIN_RECORDS, MODEL_ID, image_digest, validate_dataset` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "iNaturalist CC0 bird photographs (six species, 10 each)"
CORPUS_RELEASE = "iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (fleet DINOv2/ViT corpus subset)"
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = "CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)"
CORPUS_BYTES = 5_789_324
DEFAULT_CACHE_DIR = Path("weights") / "inat-birds"
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 6, "validation": 2, "test": 2}  # per species; 6 species -> 36 / 12 / 12
CAPTION_TEMPLATE = "a photo of a {common_name} ({scientific_name}), a wild bird photographed outdoors"
SPECIES: dict[str, tuple[str, str]] = {
    "song_sparrow": ("Melospiza melodia", "Song Sparrow"),
    "chipping_sparrow": ("Spizella passerina", "Chipping Sparrow"),
    "white_throated_sparrow": ("Zonotrichia albicollis", "White-throated Sparrow"),
    "dark_eyed_junco": ("Junco hyemalis", "Dark-eyed Junco"),
    "house_finch": ("Haemorhous mexicanus", "House Finch"),
    "american_goldfinch": ("Spinus tristis", "American Goldfinch"),
}
# (id, label, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served
#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension
#  (jpg or jpeg); the digest pins the served bytes
SAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (
    (
        "song_sparrow-00",
        "song_sparrow",
        129376982,
        79016324,
        "andywilson",
        43427,
        "7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d",
        "jpg",
    ),
    (
        "song_sparrow-01",
        "song_sparrow",
        480991086,
        267636534,
        "lyneisfilm",
        162073,
        "11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec",
        "jpg",
    ),
    (
        "song_sparrow-02",
        "song_sparrow",
        546060381,
        302980489,
        "swpollinators",
        27899,
        "4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6",
        "jpg",
    ),
    (
        "song_sparrow-03",
        "song_sparrow",
        308625896,
        177450028,
        "radrat",
        70961,
        "1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5",
        "jpeg",
    ),
    (
        "song_sparrow-04",
        "song_sparrow",
        494793016,
        275349085,
        "k-simpkins",
        58410,
        "255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e",
        "jpg",
    ),
    (
        "song_sparrow-05",
        "song_sparrow",
        674054489,
        369029444,
        "ben142",
        220573,
        "1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123",
        "jpg",
    ),
    (
        "song_sparrow-06",
        "song_sparrow",
        339623726,
        193339933,
        "rawcomposition",
        25012,
        "d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b",
        "jpg",
    ),
    (
        "song_sparrow-07",
        "song_sparrow",
        222768957,
        130949329,
        "davidfbird",
        110773,
        "0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff",
        "jpg",
    ),
    (
        "song_sparrow-08",
        "song_sparrow",
        181658744,
        107953669,
        "gcart043",
        98482,
        "a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4",
        "jpeg",
    ),
    (
        "song_sparrow-09",
        "song_sparrow",
        148994242,
        90171417,
        "glennberry",
        102702,
        "64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa",
        "jpg",
    ),
    (
        "chipping_sparrow-00",
        "chipping_sparrow",
        198992636,
        117809422,
        "k-simpkins",
        62563,
        "cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf",
        "jpg",
    ),
    (
        "chipping_sparrow-01",
        "chipping_sparrow",
        248210057,
        144599194,
        "w_mark_c",
        193389,
        "497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150",
        "jpg",
    ),
    (
        "chipping_sparrow-02",
        "chipping_sparrow",
        156350853,
        94266719,
        "ellyne",
        142332,
        "6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2",
        "jpeg",
    ),
    (
        "chipping_sparrow-03",
        "chipping_sparrow",
        16128796,
        11327134,
        "reuvenm",
        70532,
        "5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c",
        "jpeg",
    ),
    (
        "chipping_sparrow-04",
        "chipping_sparrow",
        391300648,
        220982684,
        "carterdorscht",
        208567,
        "c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21",
        "jpeg",
    ),
    (
        "chipping_sparrow-05",
        "chipping_sparrow",
        339456083,
        193252042,
        "rawcomposition",
        46329,
        "41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496",
        "jpg",
    ),
    (
        "chipping_sparrow-06",
        "chipping_sparrow",
        40457652,
        26076708,
        "andywilson",
        156894,
        "b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6",
        "jpeg",
    ),
    (
        "chipping_sparrow-07",
        "chipping_sparrow",
        84151914,
        52921135,
        "davidfbird",
        145714,
        "ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9",
        "jpeg",
    ),
    (
        "chipping_sparrow-08",
        "chipping_sparrow",
        523674610,
        291074747,
        "rwp84",
        47983,
        "41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800",
        "jpg",
    ),
    (
        "chipping_sparrow-09",
        "chipping_sparrow",
        292695018,
        168861389,
        "tim_kirsten",
        64812,
        "cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d",
        "jpeg",
    ),
    (
        "white_throated_sparrow-00",
        "white_throated_sparrow",
        339621218,
        193338380,
        "rawcomposition",
        31152,
        "d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6",
        "jpg",
    ),
    (
        "white_throated_sparrow-01",
        "white_throated_sparrow",
        166821399,
        99992799,
        "dziakj1",
        125954,
        "c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852",
        "jpeg",
    ),
    (
        "white_throated_sparrow-02",
        "white_throated_sparrow",
        469820434,
        261505977,
        "joy4birds",
        111767,
        "7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb",
        "jpg",
    ),
    (
        "white_throated_sparrow-03",
        "white_throated_sparrow",
        99351488,
        62040646,
        "bradenjudson",
        23399,
        "e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e",
        "jpeg",
    ),
    (
        "white_throated_sparrow-04",
        "white_throated_sparrow",
        177497964,
        105746665,
        "andywilson",
        29827,
        "65475d4842396f2488167453192d4aa834d2f1b40bcc78b420878c55ccf694d7",
        "jpeg",
    ),
    (
        "white_throated_sparrow-05",
        "white_throated_sparrow",
        628148203,
        344731686,
        "lavenderdame",
        106872,
        "36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8",
        "jpg",
    ),
    (
        "white_throated_sparrow-06",
        "white_throated_sparrow",
        104660609,
        65043951,
        "allan7",
        42443,
        "10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635",
        "jpeg",
    ),
    (
        "white_throated_sparrow-07",
        "white_throated_sparrow",
        250718938,
        145903421,
        "stevestevens",
        138735,
        "bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-08",
        "white_throated_sparrow",
        15105971,
        10793852,
        "schylerbrown",
        31467,
        "6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-09",
        "white_throated_sparrow",
        171460784,
        102554447,
        "w_mark_c",
        251287,
        "3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137",
        "jpg",
    ),
    (
        "dark_eyed_junco-00",
        "dark_eyed_junco",
        172110799,
        102901486,
        "schylerbrown",
        182973,
        "185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9",
        "jpeg",
    ),
    (
        "dark_eyed_junco-01",
        "dark_eyed_junco",
        46691943,
        29901256,
        "haida_gwaii",
        46823,
        "a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5",
        "jpg",
    ),
    (
        "dark_eyed_junco-02",
        "dark_eyed_junco",
        707222551,
        386266764,
        "ben142",
        289608,
        "7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89",
        "jpg",
    ),
    (
        "dark_eyed_junco-03",
        "dark_eyed_junco",
        192557376,
        114006980,
        "k-simpkins",
        172398,
        "206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479",
        "jpeg",
    ),
    (
        "dark_eyed_junco-04",
        "dark_eyed_junco",
        8793471,
        6892999,
        "truthseqr",
        45302,
        "f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2",
        "jpeg",
    ),
    (
        "dark_eyed_junco-05",
        "dark_eyed_junco",
        346777340,
        196961623,
        "zacharyfoster",
        46712,
        "ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab",
        "jpg",
    ),
    (
        "dark_eyed_junco-06",
        "dark_eyed_junco",
        274980085,
        159160633,
        "andy71",
        51209,
        "c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b",
        "jpeg",
    ),
    (
        "dark_eyed_junco-07",
        "dark_eyed_junco",
        243303909,
        141959574,
        "andywilson",
        71321,
        "ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6",
        "jpeg",
    ),
    (
        "dark_eyed_junco-08",
        "dark_eyed_junco",
        213798376,
        126031618,
        "nathanael15",
        57646,
        "d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b",
        "jpg",
    ),
    (
        "dark_eyed_junco-09",
        "dark_eyed_junco",
        63482066,
        39977347,
        "chrisleearm",
        41870,
        "90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72",
        "jpeg",
    ),
    (
        "house_finch-00",
        "house_finch",
        117990649,
        72375345,
        "kristen163",
        75945,
        "eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f",
        "jpeg",
    ),
    (
        "house_finch-01",
        "house_finch",
        389479656,
        220010434,
        "aster-asti",
        82128,
        "a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5",
        "jpg",
    ),
    (
        "house_finch-02",
        "house_finch",
        176982307,
        105476125,
        "vicki936",
        22211,
        "c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f",
        "jpeg",
    ),
    (
        "house_finch-03",
        "house_finch",
        697940852,
        381438133,
        "ben142",
        196735,
        "a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5",
        "jpg",
    ),
    (
        "house_finch-04",
        "house_finch",
        72470599,
        45698380,
        "henrya",
        61724,
        "1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b",
        "jpeg",
    ),
    (
        "house_finch-05",
        "house_finch",
        98576538,
        61594129,
        "enspring",
        46784,
        "8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c",
        "jpg",
    ),
    (
        "house_finch-06",
        "house_finch",
        80751781,
        50842166,
        "leahmfulton",
        44269,
        "6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7",
        "jpg",
    ),
    (
        "house_finch-07",
        "house_finch",
        630196420,
        345777550,
        "truthseqr",
        149455,
        "abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c",
        "jpg",
    ),
    (
        "house_finch-08",
        "house_finch",
        214612538,
        126483167,
        "hamiltonturner",
        124116,
        "6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930",
        "jpeg",
    ),
    (
        "house_finch-09",
        "house_finch",
        213077180,
        125637342,
        "jnicat",
        25823,
        "e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8",
        "jpeg",
    ),
    (
        "american_goldfinch-00",
        "american_goldfinch",
        84579952,
        53187208,
        "glennberry",
        59673,
        "72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36",
        "jpeg",
    ),
    (
        "american_goldfinch-01",
        "american_goldfinch",
        12533322,
        9255418,
        "braincellsgone",
        55661,
        "6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc",
        "jpg",
    ),
    (
        "american_goldfinch-02",
        "american_goldfinch",
        131102823,
        80016788,
        "radrat",
        98990,
        "232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff",
        "jpeg",
    ),
    (
        "american_goldfinch-03",
        "american_goldfinch",
        175048222,
        104466897,
        "eug302",
        44231,
        "11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e",
        "jpg",
    ),
    (
        "american_goldfinch-04",
        "american_goldfinch",
        68849595,
        43390778,
        "mefisher",
        154503,
        "66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1",
        "jpg",
    ),
    (
        "american_goldfinch-05",
        "american_goldfinch",
        431916465,
        242278180,
        "k-simpkins",
        45413,
        "d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d",
        "jpg",
    ),
    (
        "american_goldfinch-06",
        "american_goldfinch",
        377136648,
        213398931,
        "nathan1177",
        66516,
        "7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438",
        "jpg",
    ),
    (
        "american_goldfinch-07",
        "american_goldfinch",
        230801384,
        135330401,
        "enspring",
        42886,
        "78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed",
        "jpeg",
    ),
    (
        "american_goldfinch-08",
        "american_goldfinch",
        660472044,
        361884286,
        "ben142",
        270599,
        "733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876",
        "jpg",
    ),
    (
        "american_goldfinch-09",
        "american_goldfinch",
        294667312,
        169935316,
        "dande",
        163470,
        "39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0",
        "jpeg",
    ),
)
SAMPLE_LABEL_SOURCE = f"{CORPUS_NAME}; {CORPUS_RELEASE}; {CORPUS_LICENSE}"


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def caption_for(label: str) -> str:
    """The template caption of a species key (the prompt the tutorial trains and generates with)."""
    if label not in SPECIES:
        raise ValueError(f"unknown species key {label!r}; expected one of {sorted(SPECIES)}")
    scientific, common = SPECIES[label]
    return CAPTION_TEMPLATE.format(common_name=common, scientific_name=scientific)


def photo_url(photo_id: int, ext: str = "jpg") -> str:
    """The served object for a pinned photo; `ext` is its recorded original extension (jpg or jpeg)."""
    if ext not in ("jpg", "jpeg"):
        raise ValueError(f"unsupported photo extension {ext!r}")
    return f"{CORPUS_BASE_URL}{photo_id}/medium.{ext}"


def observation_url(observation_id: int) -> str:
    return f"https://www.inaturalist.org/observations/{observation_id}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:
        local = cache / f"{photo_id}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = photo_url(photo_id, ext)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(url, headers={"User-Agent": "kandinsky-inpainting-tutorial/1.0"})
                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 "
                    f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified photo bytes into `{id, image, caption, label}` records with their provenance."""
    from PIL import Image

    out = []
    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "caption": caption_for(label),
                "label": label,
                "scientific_name": SPECIES[label][0],
                "common_name": SPECIES[label][1],
                "inat_photo_id": photo_id,
                "inat_observation_url": observation_url(obs_id),
                "observer": user,
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified draw per species: `sizes` counts per class for train / validation / test."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_label.setdefault(str(record["label"]), []).append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        needed = sum(sizes.values())
        if len(pool) < needed:
            raise ValueError(f"{label}: only {len(pool)} records available, need {needed}")
        cursor = 0
        for name, per_class in sizes.items():
            out[name].extend(pool[cursor : cursor + per_class])
            cursor += per_class
    for name in out:
        rng.shuffle(out[name])
        out[name] = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(out[name])]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes)


def sample_prompts(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The distinct captions of a split, in first-seen order (the prompts generation and scoring use)."""
    return list(dict.fromkeys(str(r["caption"]) for r in records))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test, grouped by caption, after de-duplicating
    images. Every caption keeps at least one test record when it has three or more images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_caption: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_caption.setdefault(record["caption"], []).append(record)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for caption in sorted(by_caption):
        pool = by_caption[caption]
        rng.shuffle(pool)
        n_test = max(1, round(len(pool) * test_fraction)) if len(pool) >= 3 else 0
        n_val = round(len(pool) * val_fraction) if len(pool) >= 3 else 0
        splits["test"].extend(pool[:n_test])
        splits["validation"].extend(pool[n_test : n_test + n_val])
        splits["train"].extend(pool[n_test + n_val :])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_TRAIN_RECORDS:
        raise ValueError(f"split leaves {len(splits['train'])} training records; at least {MIN_TRAIN_RECORDS} are required")
    if not splits["test"]:
        raise ValueError("split leaves no test record; give at least one caption three or more images")
    return splits


BYOD_COLUMNS = ("id", "file", "caption")
BYOD_OPTIONAL_COLUMNS = ("mask",)


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read inpainting records from a directory or a zip holding `captions.csv` beside the image files.

    `captions.csv` has the columns `id`, `file` and `caption`, and an optional `mask` column naming a mask image
    (same width and height as its image; white or 255 = region to repaint, black or 0 = region to keep). A row whose
    `mask` is empty, or a table without the column, gets no `mask_image`, and `validate_dataset` then assigns the
    deterministic centre mask (`create_center_mask`: the middle half of each side). Files are decoded in memory and
    never extracted to disk. Every refusal names the row, the id and the file so it can be fixed in the table.
    """
    from PIL import Image, UnidentifiedImageError

    source = Path(path)
    if source.is_dir():
        table = (source / "captions.csv").read_text(encoding="utf-8") if (source / "captions.csv").is_file() else None
        names = {p.name for p in source.iterdir() if p.is_file()}
        opener = lambda name: (source / name).read_bytes()  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist() if not n.endswith("/")}
        table = archive.read(members["captions.csv"]).decode("utf-8") if "captions.csv" in members else None
        names = set(members)
        opener = lambda name: archive.read(members[name])  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding captions.csv and the image files")
    if table is None:
        raise ValueError("BYOD dataset must contain captions.csv with the columns id, file, caption (and optionally mask)")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = set(BYOD_COLUMNS) - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"captions.csv is missing columns {sorted(missing)}; required: {list(BYOD_COLUMNS)}, optional: mask")

    def decode(name: str, *, row: int, rid: str, role: str, mode: str) -> Any:
        if name not in names:
            raise ValueError(f"captions.csv row {row} (id {rid!r}): {role} file {name!r} is not in the dataset")
        try:
            image = Image.open(io.BytesIO(opener(name)))
            image.load()
        except (UnidentifiedImageError, OSError) as exc:
            message = f"captions.csv row {row} (id {rid!r}): {role} file {name!r} is not a readable image ({exc})"
            raise ValueError(message) from exc
        return image.convert(mode)

    out = []
    for row_number, row in enumerate(rows, start=2):
        rid = (row.get("id") or "").strip()
        record: dict[str, Any] = {
            "id": rid,
            "image": decode((row.get("file") or "").strip(), row=row_number, rid=rid, role="image", mode="RGB"),
            "caption": row.get("caption") or "",
        }
        mask_name = (row.get("mask") or "").strip()
        if mask_name:
            mask = decode(mask_name, row=row_number, rid=rid, role="mask", mode="L")
            if mask.size != record["image"].size:
                raise ValueError(
                    f"captions.csv row {row_number} (id {rid!r}): mask {mask_name!r} is {mask.size[0]}x{mask.size[1]} px "
                    f"but its image is {record['image'].size[0]}x{record['image'].size[1]} px; they must match"
                )
            record["mask_image"] = mask
        out.append(record)
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the captions table of a split (id, file, caption, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=["id", "file", "caption", "mask", "label", "observer", "inat_observation_url"]
        )
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "file": f"{record['inat_photo_id']}.jpg" if record.get("inat_photo_id") else f"{record['id']}.jpg",
                    "caption": record["caption"],
                    "mask": record.get("mask_file", ""),
                    "label": record.get("label", ""),
                    "observer": record.get("observer", ""),
                    "inat_observation_url": record.get("inat_observation_url", ""),
                }
            )
    return out


def dataset_manifest(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Validate every split and summarise the dataset (counts, captions, digests) for provenance exports."""
    summary: dict[str, Any] = {"model_id": MODEL_ID, "splits": {}}
    for name, records in splits.items():
        report = validate_dataset(records, min_records=1)
        summary["splits"][name] = {
            "n_records": report["n_records"],
            "n_captions": report["n_captions"],
            "shorter_side": report["shorter_side"],
            "centre_cropped": report["centre_cropped"],
            "digest": report["digest"],
        }
    summary["disjoint"] = check_split_disjoint(splits)
    digests = json.dumps({k: v["digest"] for k, v in summary["splits"].items()}, sort_keys=True)
    summary["digest"] = hashlib.sha256(digests.encode("utf-8")).hexdigest()
    return summary

## 3. Pin, stage and verify the model · [Engineering]

> **Infrastructure.** The code cell(s) in this section are collapsed. You may run them without studying their implementation; they set up a reproducible, verified environment and are not learning objectives.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `db790ad5cbca…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `KandinskyInpaintPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, prior_dir=DEFAULT_PRIOR_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), use_lora=True)` load the verified files. There is no fallback to a different download and no remote model code is executed. The package also pins a second snapshot `kandinsky-2-2-prior` (14 files), a second snapshot `clip-vit-b-32-laion2b` (9 files), carried and verified the same way. The effective identity, device and weight source are printed before any inference.

**Expected result:** the identity, the files fetched on a fresh runtime (an empty list when the snapshot is already staged), the verified file counts, and the device the pipeline loaded on.

In [ ]:
# @title Infrastructure: pin, stage and digest-verify the snapshots, then load the pipeline
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "kandinsky-2-2-decoder-inpaint",
  "modelId": "kandinsky-community/kandinsky-2-2-decoder-inpaint",
  "revision": "db790ad5cbcabed886f069ef2710774657621702",
  "license": "apache-2.0",
  "totalBytes": 5283767688,
  "files": [
    {
      "path": "README.md",
      "bytes": 5720,
      "sha256": "2d4ec36bbaf91ab7d1d1d2ee3a921d5c16f39e8d1ce6d3f9045a8d82cb0ef186"
    },
    {
      "path": "model_index.json",
      "bytes": 257,
      "sha256": "6dfda60c9ec616ddd4f38cdf01a2adf3e1c6cf074e925f7bb77980d543aee8b1"
    },
    {
      "path": "movq/config.json",
      "bytes": 660,
      "sha256": "b7448f3ea8aecf233629b20c1f606887e22244656edcee1be6a419079717a23e"
    },
    {
      "path": "movq/diffusion_pytorch_model.safetensors",
      "bytes": 271380364,
      "sha256": "43a5860fea195a7116f2471396c5cc9535fade9b63c4857d8a192ffd924b7002"
    },
    {
      "path": "scheduler/scheduler_config.json",
      "bytes": 318,
      "sha256": "801b60ec20835a36bf8c0cd2dab9f2c72d0b938ec87202a6b5b3eeb2714e1b97"
    },
    {
      "path": "unet/config.json",
      "bytes": 1665,
      "sha256": "35e05e11683958668f34f2d99fb63524c26863f5d21257fda3c4a684e9f3131d"
    },
    {
      "path": "unet/diffusion_pytorch_model.safetensors",
      "bytes": 5012378704,
      "sha256": "098b846d2378b4b44a33e0bd47f89f2886b72a10200cbf9e96d5fae3c471543f"
    }
  ]
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})

PRIOR_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "kandinsky-2-2-prior",
  "modelId": "kandinsky-community/kandinsky-2-2-prior",
  "revision": "9fc51ad5732afc5d031724219d22e6c42179c5a8",
  "files": [
    {
      "path": "README.md",
      "bytes": 14444,
      "sha256": "bba4898931a219c1d2f2f832bb8050e205a94ec781e7ebee55eb96c89add286d"
    },
    {
      "path": "model_index.json",
      "bytes": 501,
      "sha256": "26cba76e560b6da612d65b8e7beac9be04442df8df2e1743b8466454988d1f5e"
    },
    {
      "path": "text_encoder/config.json",
      "bytes": 2017,
      "sha256": "3dec84fa855c0b7f531eea433e805a1194086c38c5635baceda8cbed4f727f1b"
    },
    {
      "path": "text_encoder/model.safetensors",
      "bytes": 2778702976,
      "sha256": "6eaafeea664d7ab87df17274d6114dfdf092c587c7cd56c13cf84aacdebc0332"
    },
    {
      "path": "image_processor/preprocessor_config.json",
      "bytes": 315,
      "sha256": "0a70aaa5de2ccb5e222b0e7e3ea347a3da051c3588c763cdd23b4d80bbf96b65"
    },
    {
      "path": "tokenizer/merges.txt",
      "bytes": 524619,
      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"
    },
    {
      "path": "tokenizer/special_tokens_map.json",
      "bytes": 389,
      "sha256": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf"
    },
    {
      "path": "tokenizer/tokenizer_config.json",
      "bytes": 904,
      "sha256": "e19f34ef773563fb695f96cfcae1e4c7b112ab6ad532f6962061df5242d924f0"
    },
    {
      "path": "tokenizer/vocab.json",
      "bytes": 862328,
      "sha256": "5047b556ce86ccaf6aa22b3ffccfc52d391ea4accdab9c2f2407da5b742d4363"
    },
    {
      "path": "image_encoder/config.json",
      "bytes": 2013,
      "sha256": "1d53c2b4b74c5f85171d313adda3e3b8771ff5c698ee66a29710d0ac822298e4"
    },
    {
      "path": "image_encoder/model.safetensors",
      "bytes": 3689912664,
      "sha256": "657723e09f46a7c3957df651601029f66b1748afb12b419816330f16ed45d64d"
    },
    {
      "path": "scheduler/scheduler_config.json",
      "bytes": 229,
      "sha256": "5ba056e705a3317f60d79a3756a3409b0a53f1129b76a6aa5097ce9f513d143e"
    },
    {
      "path": "prior/config.json",
      "bytes": 252,
      "sha256": "e36f9ae41677b87a8ca72853f1246704228c49c280adcfa8a8e100ef47d2ee5a"
    },
    {
      "path": "prior/diffusion_pytorch_model.safetensors",
      "bytes": 4104940968,
      "sha256": "ab139b2df9e40ed0b1eda8dc6270a139a796c5fa3af76cd6b0573f620dca6dde"
    }
  ],
  "totalBytes": 10574964619
}

if (PRIOR_MANIFEST['modelId'], PRIOR_MANIFEST['revision']) != (PRIOR_ID, PRIOR_REVISION):
    raise RuntimeError('inline kandinsky-2-2-prior manifest does not name the identity carried by the pipeline module')
DEFAULT_PRIOR_DIR.mkdir(parents=True, exist_ok=True)
with open(DEFAULT_PRIOR_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(PRIOR_MANIFEST, handle, indent=2)
fetched_kandinsky_2_2_prior = stage_missing_prior_files(DEFAULT_PRIOR_DIR, allow_download=True)
print({'weights_dir': str(DEFAULT_PRIOR_DIR), 'fetched': fetched_kandinsky_2_2_prior})
_extra = verify_prior_snapshot(DEFAULT_PRIOR_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_kandinsky_2_2_prior': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})

SCORER_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "clip-vit-b-32-laion2b",
  "modelId": "laion/CLIP-ViT-B-32-laion2B-s34B-b79K",
  "revision": "1a25a446712ba5ee05982a381eed697ef9b435cf",
  "files": [
    {
      "path": "README.md",
      "bytes": 7464,
      "sha256": "d0600101ba66849bd84eeae14e0b1ccd56cd962ea94aec5d60c415d9ce4e5f28"
    },
    {
      "path": "config.json",
      "bytes": 4355,
      "sha256": "1284cbff35169abb23a1c5525a8b0f543c7bd191d4b9aed63880c1571bc4191c"
    },
    {
      "path": "merges.txt",
      "bytes": 524618,
      "sha256": "f1cd414c2d113a28b296afc0fd66ae854c0e0c34d5de373a2f5b7f62e0c02344"
    },
    {
      "path": "model.safetensors",
      "bytes": 605157884,
      "sha256": "74813fbcdc750f235c9784c367ca1394d2a5c25eb0aac92761752ac239db7cff"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 316,
      "sha256": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 389,
      "sha256": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf"
    },
    {
      "path": "tokenizer.json",
      "bytes": 2224041,
      "sha256": "b556ac8c99757ffb677208af34bc8c6721572114111a6e0aaf5fa69ff0b8d842"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 904,
      "sha256": "e19f34ef773563fb695f96cfcae1e4c7b112ab6ad532f6962061df5242d924f0"
    },
    {
      "path": "vocab.json",
      "bytes": 862328,
      "sha256": "5047b556ce86ccaf6aa22b3ffccfc52d391ea4accdab9c2f2407da5b742d4363"
    }
  ],
  "totalBytes": 608782299
}

if (SCORER_MANIFEST['modelId'], SCORER_MANIFEST['revision']) != (SCORER_ID, SCORER_REVISION):
    raise RuntimeError('inline clip-vit-b-32-laion2b manifest does not name the identity carried by the pipeline module')
DEFAULT_SCORER_DIR.mkdir(parents=True, exist_ok=True)
with open(DEFAULT_SCORER_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(SCORER_MANIFEST, handle, indent=2)
fetched_clip_vit_b_32_laion2b = stage_missing_scorer_files(DEFAULT_SCORER_DIR, allow_download=True)
print({'weights_dir': str(DEFAULT_SCORER_DIR), 'fetched': fetched_clip_vit_b_32_laion2b})
_extra = verify_scorer_snapshot(DEFAULT_SCORER_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_clip_vit_b_32_laion2b': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})
pipe = KandinskyInpaintPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, prior_dir=DEFAULT_PRIOR_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), use_lora=True)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample photographs, masks, validation and splits · [Concept]

The default dataset is 60 research-grade iNaturalist photographs of six common North American birds — 10 per species, one per observer per species, every one CC0 — fetched by photo id from the open-data bucket and refused on any byte-size or SHA-256 mismatch (`fetch_corpus`). Each caption is generated from the species by one template. `build_sample_dataset` draws a seeded stratified split — 6 / 2 / 2 per species for training, validation and test. The sample has no hand-drawn masks, so `validate_dataset` gives every record the deterministic **centre mask**: a white box over the middle half of each side, which covers a quarter of the photograph and usually most of the bird. `dataset_manifest` validates every split, checks that no photograph appears twice and records a digest.

**Predict before running:** the centre mask covers the middle half of each side. What fraction of each 512 × 512 photograph will be repainted? Which of the three probes below do you expect validation to refuse?

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    if BYOD_PATH:
        byod_path = Path(BYOD_PATH)
    else:
        from google.colab import files
        uploaded = files.upload()
        file_name, payload = next(iter(uploaded.items()))
        byod_path = Path('work') / file_name
        byod_path.parent.mkdir(parents=True, exist_ok=True)
        byod_path.write_bytes(payload)
    byod_records = load_byod_dataset(byod_path)
    own_masks = sum('mask_image' in r for r in byod_records)
    print({'byod_records': len(byod_records), 'own_masks': own_masks, 'centre_mask_fallback': len(byod_records) - own_masks})
    splits = split_dataset(byod_records, seed=0)
    data_source = 'BYOD (' + byod_path.name + ')'
else:
    splits = fetch_sample_dataset(cache_dir='weights/inat-birds')
    data_source = SAMPLE_LABEL_SOURCE
# validate_dataset attaches the mask: a record's own mask, or the deterministic centre mask when it has none
train_records, val_records, test_records = (validate_dataset(splits[name], min_records=1)['records'] for name in ('train', 'validation', 'test'))

dataset_report = dataset_manifest({'train': train_records, 'validation': val_records, 'test': test_records})
print({'data_source': data_source, 'splits': {k: v['n_records'] for k, v in dataset_report['splits'].items()}, 'captions': dataset_report['splits']['train']['n_captions'], 'disjoint': dataset_report['disjoint']})
print({'shorter_side': dataset_report['splits']['train']['shorter_side'], 'centre_cropped': dataset_report['splits']['train']['centre_cropped'], 'digest': dataset_report['digest'][:16] + '...'})
_, first_mask = preprocess_image_and_mask(test_records[0]['image'], test_records[0]['mask_image'])
print({'first_test_record': validate_inputs(test_records[0]), 'caption': test_records[0]['caption'], 'repaint_fraction': round(float((np.asarray(first_mask) > 0).mean()), 4)})
prompts = sample_prompts(train_records)
print({'prompts': prompts})
sample_csv = write_dataset_csv(test_records, 'outputs/kandinsky_inpainting_sample_captions.csv')
print({'sample_csv': str(sample_csv)})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'missing caption': [{'id': r['id'], 'image': r['image']} for r in train_records[:4]],
    'image too small': [{**train_records[0], 'image': Image.new('RGB', (200, 200))}, *train_records[1:4]],
    'mask size differs from image': [{**train_records[0], 'mask_image': Image.new('L', (300, 300))}, *train_records[1:4]],
    'duplicate id': [train_records[0], *train_records[:4]],
}
for name, records in probes.items():
    try:
        validate_dataset(records)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

**What to notice:** 36 / 12 / 12 records and six distinct captions on the sample path, `'disjoint'` listing the same counts, a shorter side of a few hundred pixels (every photograph is resized and centre-cropped to 512²), a `repaint_fraction` of 0.25, a written `outputs/kandinsky_inpainting_sample_captions.csv` in the shape BYOD expects (its `mask` column is empty because the sample uses the centre mask), and every probe rejected before any model runs, each message naming the failed rule.

<details>
<summary>Check your reasoning (open after answering)</summary>

The mask box spans the middle half of the width and the middle half of the height, so it covers 0.5 × 0.5 = 0.25 of the pixels. All four probes are refused: a missing caption and a duplicate id break the record schema, a 200 px side is below the 256 px minimum, and a mask of a different size cannot be aligned with its photograph. Validation never looks at *content*: a mask over the sky and a caption that describes another bird would both pass.

</details>

## 5. Encode every caption with the prior, then release it · [Concept]

`pipe.encode_prompts` loads the Kandinsky 2.2 diffusion prior from the verified snapshot (float16 on CUDA), runs the prior's own diffusion process to produce a CLIP image embedding for each distinct caption, and keeps the embeddings on the CPU. Each caption's prior run is seeded with `prompt_seed(caption)`, a 31-bit number taken from the caption's SHA-256, so the same caption gets the same seed in every process. Encoded here: the six captions (training, validation and test share them), one new caption for Section 9, and the empty negative caption that classifier-free guidance needs. `release_prior` then frees the prior's GPU memory.

**Expected result:** eight captions encoded (seven on the sample path plus the empty one), and GPU memory falling after the release.

In [ ]:
import time

NEW_PROMPT = 'a photo of a House Finch (Haemorhous mexicanus) perched on a snow-covered branch in winter'

def gpu_memory_gb():
    return round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None

all_prompts = sample_prompts(train_records + val_records + test_records) + [NEW_PROMPT]
encode_report = pipe.encode_prompts(all_prompts)
print({**encode_report, 'gpu_memory_gb_with_prior': gpu_memory_gb()})
print({'prompt_seed': {p[:48]: prompt_seed(p) for p in all_prompts[:2]}})
released = pipe.release_prior()
print({'prior_released': released, 'gpu_memory_gb_after_release': gpu_memory_gb(), 'device': pipe.device, 'precision': str(pipe.dtype).replace('torch.', '')})

## 6. The frozen baseline: denoising loss, preservation and CLIP · [Evaluation practice]

**Question tested:** before any training, how well does the pretrained model predict noise on held-out photographs, how well does it preserve the kept region, and how well do its repainted photographs match their captions? The pipeline was built with `use_lora=True`, but the adapter's B matrices start at zero, so until Section 7 this is the pretrained model.

- **Held-out denoising loss** (`pipe.evaluate`): each held-out photograph is MoVQ-encoded, noised at five fixed timesteps (100, 300, 500, 700, 900) with seeded noise, and the UNet — given the masked latents and the keep mask — predicts that noise; the score is the mean squared error. The same seed gives the same latents, noise and timesteps later, so the adapted number is a paired comparison, not a re-draw.
- **Preservation** (`score_inpainting_preservation`): PSNR in dB and SSIM between the input and the output, computed only over the kept (black) region. The UNet never repaints that region, but the output is decoded by the MoVQ, so preservation measures how faithfully the kept region survives the encode-decode round trip. The SSIM here is one global statistic over the kept pixels, not the windowed SSIM of image-quality papers.
- **CLIP prompt similarity** (`score_generations`): the pinned CLIP ViT-B/32 scores each whole output against its caption. Two references frame it: a **mean-fill floor**, where the repaint region is filled with the mean colour of the kept region (no model at all), and the **original-photograph ceiling**, the unmasked photograph scored the same way.

**Predict before running:** will the frozen model's CLIP similarity sit closer to the mean-fill floor or to the original-photograph ceiling? Will preservation be perfect (PSNR above 60 dB) or visibly imperfect?

In [ ]:
STEPS = 20  # @param {type:"integer"}
GUIDANCE_SCALE = 4.0  # @param {type:"number"}
EVAL_SEED = 0
GENERATION_SEED = 1000

def prepared(records):
    pairs = [preprocess_image_and_mask(r['image'], r['mask_image']) for r in records]
    return [image for image, _ in pairs], [mask for _, mask in pairs]

def mean_fill(image, mask):
    pixels = np.asarray(image).copy()
    repaint = np.asarray(mask) > 0
    pixels[repaint] = pixels[~repaint].mean(axis=0).astype(np.uint8)
    return Image.fromarray(pixels)

def masked_view(image, mask):
    pixels = np.asarray(image).copy()
    pixels[np.asarray(mask) > 0] = 128
    return Image.fromarray(pixels)

def triptych_grid(images, masks, outputs, path, rows=6):
    sheet = Image.new('RGB', (256 * 3, 256 * min(rows, len(images))), 'white')
    for i, (image, mask, output) in enumerate(list(zip(images, masks, outputs))[:rows]):
        for j, tile in enumerate((image, masked_view(image, mask), output)):
            sheet.paste(tile.resize((256, 256)), (256 * j, 256 * i))
    sheet.save(path)
    return path

def clip_mean(scores):
    return round(scores['mean_prompt_similarity'], 3)

if pipe.adapter is not None:
    raise RuntimeError('this pipeline is already adapted, so Section 6 would not measure the frozen model: re-run from Section 3 (Runtime → Run cell and below)')
scorer = ClipScorer(device=pipe.device, weights_dir=str(DEFAULT_SCORER_DIR))
t0 = time.perf_counter()
frozen_val = pipe.evaluate(val_records, seed=EVAL_SEED)
frozen_test = pipe.evaluate(test_records, seed=EVAL_SEED)
print({'frozen_denoising_mse': {'validation': frozen_val['denoising_mse'], 'test': frozen_test['denoising_mse']}, 'by_timestep_test': frozen_test['by_timestep'], 'seconds': round(time.perf_counter() - t0, 1)})

test_images, test_masks = prepared(test_records)
test_prompts = [r['caption'] for r in test_records]
frozen_generation = pipe.generate(test_records, seed=GENERATION_SEED, steps=STEPS, guidance_scale=GUIDANCE_SCALE)
frozen_images = [g['image'] for g in frozen_generation['results']]
print({'inpainted': len(frozen_images), 'steps': frozen_generation['steps'], 'guidance_scale': frozen_generation['guidance_scale'], 'seconds': frozen_generation['seconds'], 'adapted': frozen_generation['model']['adapted']})
frozen_preservation = score_inpainting_preservation(test_images, frozen_images, test_masks)
frozen_clip = score_generations(scorer, frozen_images, test_prompts)
fill_clip = score_generations(scorer, [mean_fill(i, m) for i, m in zip(test_images, test_masks)], test_prompts)
real_clip = score_generations(scorer, test_images, test_prompts)
print({'clip_prompt_similarity': {'mean_fill_floor': clip_mean(fill_clip), 'frozen': clip_mean(frozen_clip), 'original_photo_ceiling': clip_mean(real_clip)}})
print({'frozen_preservation': {'mean_unmasked_psnr_db': round(frozen_preservation['mean_unmasked_psnr_db'], 2), 'mean_unmasked_ssim': round(frozen_preservation['mean_unmasked_ssim'], 4)}})
for result, clip in list(zip(frozen_generation['results'], frozen_clip['prompt_similarities']))[:6]:
    print({'id': result['id'], 'caption': result['prompt'][:40], 'unmasked_psnr_db': result['unmasked_psnr_db'], 'unmasked_ssim': result['unmasked_ssim'], 'clip': round(clip, 2)})
print({'grid': str(triptych_grid(test_images, test_masks, frozen_images, 'outputs/kandinsky_inpainting_frozen_grid.jpg'))})

**What to notice:** the denoising loss by timestep — noise is harder to predict at low timesteps (little noise) than the mean suggests, so compare timestep by timestep later. The three CLIP numbers should be ordered, with the floor lowest; the grid shows each photograph, its masked view and the frozen output side by side. Preservation is high but not perfect.

<details>
<summary>Check your reasoning (open after answering)</summary>

A mean-colour fill leaves most of the bird out, so CLIP usually scores it well below the original photograph; a working inpainting model lands between the two, often close to the ceiling because three quarters of the photograph are kept. Preservation is not perfect because the kept region is not copied pixel for pixel: it is carried through the MoVQ latents and decoded, which blurs fine texture slightly. A PSNR in the high 20s or 30s dB is a round-trip loss, not the model repainting the kept region. None of these numbers says whether a person would find the repainted bird convincing.

</details>

## 7. Bounded LoRA fine-tuning · [Concept]

`pipe.adapt` trains the 176 LoRA tensors (rank 8, 1,646,592 parameters — 0.13 % of the UNet) that `peft` attached to the UNet's attention projections `to_q`, `to_k`, `to_v` and `to_out.0`, and nothing else; the base UNet, the MoVQ and the prior are frozen. This is gradient-based parameter-efficient fine-tuning, not in-context conditioning. Each step takes one training photograph's latents and keep mask (MoVQ-encoded once), draws a timestep uniformly from the 1,000-step schedule and a noise tensor (both seeded), adds the noise, and minimises the mean squared error between the predicted and the true noise. The optimiser is AdamW at a fixed learning rate, with gradient-norm clipping at 1.0; on CUDA the LoRA tensors stay in float32 while the forward pass uses float16 autocast with a gradient scaler. Epoch 0 records the frozen model's validation loss, and the epoch with the lowest validation denoising loss is kept.

**Predict before running:** will the training loss fall steadily from epoch to epoch? Will the validation loss fall by more or less than the training loss?

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 1  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_denoising_mse': entry['val_loss']}
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=EVAL_SEED, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['adapter']['n_trainable'], 'total_parameters': adapt_result['adapter']['n_total'], 'steps': adapt_result['steps'], 'best_epoch': adapt_result['best_epoch'], 'optimizer': adapt_result['adapter']['optimizer'], 'precision': adapt_result['adapter']['precision'], 'seconds': adapt_seconds, 'gpu_memory_gb': gpu_memory_gb()})

**What to notice:** one row per epoch, epoch 0 with no training loss; a training loss that jumps between epochs because each step draws a random timestep; and a `best_epoch` that may be 0 if no epoch improved the validation loss — in that case the kept adapter is the untrained one.

<details>
<summary>Check your reasoning (open after answering)</summary>

The training loss is noisy, not a smooth curve: with one photograph per step, a step at a low timestep and a step at a high timestep have very different losses. The validation loss is measured at five fixed timesteps with fixed noise, so it moves more smoothly and usually by a smaller amount. A falling validation loss shows the adapter predicts noise better on unseen photographs of the same birds; it does not show the repainted birds look better.

</details>

## 8. Held-out evaluation: the paired comparison · [Evaluation practice]

**Question tested:** on photographs never used for training or for choosing the epoch, what changed? The adapted model is measured exactly as the frozen model was in Section 6 — the same seed, so the same latents, noise and timesteps for the denoising loss, and the same photographs, masks and generation seeds for inpainting. The cell asserts only what the procedure guarantees: the kept epoch's validation loss is no higher than the frozen model's (epoch 0), and re-measuring it reproduces the history. The test-split change is printed, not asserted.

**Predict before running:** will the test denoising loss fall? Will CLIP prompt similarity move towards the ceiling? Will preservation change, and why or why not?

In [ ]:
adapted_val = pipe.evaluate(val_records, seed=EVAL_SEED)
adapted_test = pipe.evaluate(test_records, seed=EVAL_SEED)
adapted_generation = pipe.generate(test_records, seed=GENERATION_SEED, steps=STEPS, guidance_scale=GUIDANCE_SCALE)
adapted_images = [g['image'] for g in adapted_generation['results']]
adapted_preservation = score_inpainting_preservation(test_images, adapted_images, test_masks)
adapted_clip = score_generations(scorer, adapted_images, test_prompts)
comparison = {
    'denoising_mse_validation': {'frozen': frozen_val['denoising_mse'], 'adapted': adapted_val['denoising_mse']},
    'denoising_mse_test': {'frozen': frozen_test['denoising_mse'], 'adapted': adapted_test['denoising_mse']},
    'denoising_mse_test_by_timestep': {t: {'frozen': frozen_test['by_timestep'][t], 'adapted': adapted_test['by_timestep'][t]} for t in adapted_test['by_timestep']},
    'mean_unmasked_psnr_db': {'frozen': round(frozen_preservation['mean_unmasked_psnr_db'], 2), 'adapted': round(adapted_preservation['mean_unmasked_psnr_db'], 2)},
    'mean_unmasked_ssim': {'frozen': round(frozen_preservation['mean_unmasked_ssim'], 4), 'adapted': round(adapted_preservation['mean_unmasked_ssim'], 4)},
    'clip_prompt_similarity': {'mean_fill_floor': clip_mean(fill_clip), 'frozen': clip_mean(frozen_clip), 'adapted': clip_mean(adapted_clip), 'original_photo_ceiling': clip_mean(real_clip)},
}
for name, row in comparison.items():
    print({name: row})
for before, after, clip_before, clip_after in list(zip(frozen_generation['results'], adapted_generation['results'], frozen_clip['prompt_similarities'], adapted_clip['prompt_similarities']))[:6]:
    print({'id': before['id'], 'caption': before['prompt'][:40], 'clip': {'frozen': round(clip_before, 2), 'adapted': round(clip_after, 2)}, 'unmasked_psnr_db': {'frozen': before['unmasked_psnr_db'], 'adapted': after['unmasked_psnr_db']}})
print({'grid': str(triptych_grid(test_images, test_masks, adapted_images, 'outputs/kandinsky_inpainting_adapted_grid.jpg'))})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'components': {'prior': {'id': PRIOR_ID, 'revision': PRIOR_REVISION}, 'scorer': {'id': SCORER_ID, 'revision': SCORER_REVISION}},
    'data_source': data_source,
    'dataset': dataset_report,
    'mask': 'record mask, or the deterministic centre mask (middle half of each side) when a record has none',
    'generation': {'steps': STEPS, 'guidance_scale': GUIDANCE_SCALE, 'seed': GENERATION_SEED},
    'frozen': {'validation': frozen_val, 'test': frozen_test, 'preservation': frozen_preservation, 'clip': frozen_clip},
    'adapted': {'validation': adapted_val, 'test': adapted_test, 'preservation': adapted_preservation, 'clip': adapted_clip},
    'references': {'mean_fill_floor_clip': fill_clip, 'original_photo_ceiling_clip': real_clip},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result['adapter'].items() if k != 'trainable_names'},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/kandinsky_inpainting_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
best = adapt_result['history'][adapt_result['best_epoch']]
assert best['val_loss'] <= adapt_result['history'][0]['val_loss']
assert abs(adapted_val['denoising_mse'] - best['val_loss']) < 1e-4
print({'test_denoising_mse_change': round(adapted_test['denoising_mse'] - frozen_test['denoising_mse'], 6), 'note': 'held-out observation, not asserted'})
print({'report': 'outputs/kandinsky_inpainting_evaluation_report.json'})

**What to notice:** read the rows side by side. The denoising loss rows compare identical inputs; the CLIP row places both models between the floor and the ceiling; the preservation rows should barely move. Open the two grids in `outputs/` and compare the same photograph in each.

<details>
<summary>Check your reasoning (open after answering)</summary>

Preservation barely moves because the kept region is carried through the latents in both models: LoRA changes how the UNet repaints, not how the MoVQ decodes. A lower test denoising loss is the most direct evidence that the adapter learned something that transfers to unseen photographs of these birds. A CLIP change of a few tenths on twelve photographs from one seeded run is within the noise you would see from a different generation seed; treat it as an observation, not an improvement. If the frozen model already sits near the ceiling, there is little room for CLIP to rise at all.

</details>

## 9. A new caption, artifact export and fresh reload · [Engineering]

**New-data inference.** The adapted model inpaints `NEW_PROMPT` — a caption that appears in no training record — into two held-out test photographs with their centre masks. The CLIP similarity and preservation are printed as a sanity check, not an evaluation.

**Export.** `pipe.save_artifact` writes the 176 trained tensors as `adapter.safetensors` with a `manifest.json` recording the artifact format, the decoder's id and revision, the LoRA configuration, the tensor names, the optimiser and precision, the file size and SHA-256, and the metadata passed in. The base model is not in the artifact: it must be reloaded from the pinned revision.

**Fresh reload.** `KandinskyInpaintPipeline.from_artifact` re-verifies the decoder and prior snapshots, checks the artifact format, the base-model identity and the weights digest **before** loading the tensors, and builds a new UNet with the adapter attached — a new object from files, not the in-memory model. The fresh pipeline adopts the caption embeddings already encoded, and the cell asserts that it reproduces the held-out denoising loss and the same inpainted photograph for the same caption and seed within a stated tolerance.

**Expected result:** two saved PNGs, an artifact of about 6.6 MB (1,646,592 float32 values), and a reload parity with a denoising-loss difference below `1e-6` and a mean absolute pixel difference below 1.0.

In [ ]:
import platform
import shutil

new_records = [{**r, 'caption': NEW_PROMPT} for r in test_records[:2]]
new_generation = pipe.generate(new_records, seed=2000, steps=STEPS, guidance_scale=GUIDANCE_SCALE)
new_clip = score_generations(scorer, [g['image'] for g in new_generation['results']], [NEW_PROMPT] * len(new_records))
new_predictions = []
for i, (result, clip) in enumerate(zip(new_generation['results'], new_clip['prompt_similarities'])):
    path = f'outputs/kandinsky_inpainting_new_prompt_{i}.png'
    result['image'].save(path)
    new_predictions.append({'id': result['id'], 'prompt': NEW_PROMPT, 'image': path, 'unmasked_psnr_db': result['unmasked_psnr_db'], 'unmasked_ssim': result['unmasked_ssim'], 'clip_prompt_similarity': round(clip, 3)})
    print({**new_predictions[-1], 'note': 'sanity check, not an evaluation'})

artifact_dir = Path('outputs/kandinsky_inpainting_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'kandinsky_inpainting', 'data_source': data_source, 'dataset_digest': dataset_report['digest']})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': artifact_manifest['weights']['n_tensors'], 'bytes': artifact_manifest['weights']['bytes'], 'sha256': artifact_manifest['weights']['sha256'][:16] + '...'})

reloaded = KandinskyInpaintPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, prior_dir=DEFAULT_PRIOR_DIR, device=pipe.device)
reloaded.import_prompt_cache(pipe.export_prompt_cache())
reloaded_test = reloaded.evaluate(test_records, seed=EVAL_SEED)
before = pipe.generate(test_records[:1], seed=3000, steps=STEPS, guidance_scale=GUIDANCE_SCALE)['results'][0]['image']
after = reloaded.generate(test_records[:1], seed=3000, steps=STEPS, guidance_scale=GUIDANCE_SCALE)['results'][0]['image']
parity = {'denoising_mse_diff': round(abs(reloaded_test['denoising_mse'] - adapted_test['denoising_mse']), 8), 'mean_abs_pixel_diff': round(float(np.abs(np.asarray(before, dtype=np.float32) - np.asarray(after, dtype=np.float32)).mean()), 4)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['denoising_mse_diff'] < 1e-6 and parity['mean_abs_pixel_diff'] < 1.0

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'precision': str(pipe.dtype).replace('torch.', ''), 'source': pipe.source},
    'components': {**evaluation_report['components'], 'licenses': {'prior': 'apache-2.0', 'scorer': 'mit'}},
    'provenance': {
        'snapshots': {'decoder': len(MANIFEST['files']), 'prior': len(PRIOR_MANIFEST['files']), 'scorer': len(SCORER_MANIFEST['files'])},
        'safetensors_only': True,
        'remote_code_executed': False,
        'prior_released_before_training': released,
        'prompt_seed': 'SHA-256 of the caption, first 4 bytes, 31 bits',
        'data_base_url': CORPUS_BASE_URL,
        'data_license': CORPUS_LICENSE,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'diffusers': diffusers.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    'data_source': data_source,
    'comparison': comparison,
    'new_prompt_predictions': new_predictions,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['weights']['sha256'], 'bytes': artifact_manifest['weights']['bytes']},
    'reload_parity': parity,
}
with open('outputs/kandinsky_inpainting_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

**What to notice:** the reload parity is what the artifact boundary guarantees — the same tensors, loaded into a freshly verified base model, reproduce the same outputs. It does not show that the outputs are good. `outputs/kandinsky_inpainting_result.json` gathers identity, provenance, runtime versions, the comparison, the new-caption predictions and the parity in one machine-readable file.

## 10. Change one thing: the mask size · [Evaluation practice]

**Predict → Change one thing → Run → Observe → Explain.** This optional activity inpaints the first two test photographs again with the adapted model, the same captions and the same generation seeds as Section 8. Only the mask changes: a centred box whose side is a quarter (`small`) or three quarters (`large`) of each side, instead of the centre mask's half. It runs after every required output has been written and changes nothing that earlier sections produced; set `RUN_ACTIVITY = False` to skip it.

**Predict before running:** with a `large` mask, will the kept-region PSNR rise or fall? Will CLIP prompt similarity rise or fall? What do you expect with `small`?

In [ ]:
RUN_ACTIVITY = True  # @param {type:"boolean"}
ACTIVITY_MASK = 'large'  # @param ["small", "large"]

def box_mask(image, side_fraction):
    width, height = image.size
    mask = Image.new('L', (width, height), 0)
    margin_x, margin_y = round(width * (1 - side_fraction) / 2), round(height * (1 - side_fraction) / 2)
    mask.paste(255, (margin_x, margin_y, width - margin_x, height - margin_y))
    return mask

if RUN_ACTIVITY:
    side = {'small': 0.25, 'large': 0.75}[ACTIVITY_MASK]
    changed = [{**r, 'mask_image': box_mask(r['image'], side)} for r in test_records[:2]]
    activity_generation = pipe.generate(changed, seed=GENERATION_SEED, steps=STEPS, guidance_scale=GUIDANCE_SCALE)
    activity_clip = score_generations(scorer, [g['image'] for g in activity_generation['results']], test_prompts[:2])
    for i, (centre, new) in enumerate(zip(adapted_generation['results'][:2], activity_generation['results'])):
        print({'id': centre['id'], 'repaint_fraction': {'centre': 0.25, ACTIVITY_MASK: round(side * side, 4)}, 'unmasked_psnr_db': {'centre': centre['unmasked_psnr_db'], ACTIVITY_MASK: new['unmasked_psnr_db']}, 'clip': {'centre': round(adapted_clip['prompt_similarities'][i], 2), ACTIVITY_MASK: round(activity_clip['prompt_similarities'][i], 2)}})
    activity_images, activity_masks = prepared(changed)
    print({'grid': str(triptych_grid(activity_images, activity_masks, [g['image'] for g in activity_generation['results']], 'outputs/kandinsky_inpainting_activity_' + ACTIVITY_MASK + '_grid.jpg'))})

**Observe and explain:** compare each row's `centre` and changed values, then open the activity grid. Did the result match your prediction? Explain it in terms of what the model is allowed to repaint and what the metrics measure. Then switch `ACTIVITY_MASK` and run this cell again.

<details>
<summary>Check your reasoning (open after answering)</summary>

A larger mask gives the model more of the photograph to repaint, so more of the bird is generated rather than kept. CLIP compares the whole photograph with the caption, so it can move either way: a well-repainted large region can still score high, while a poor one pulls the score down. Kept-region PSNR is computed over fewer pixels with a large mask, and those pixels sit near the edges of the photograph; its change reflects the MoVQ round trip on different content, not better or worse inpainting. The lesson: preservation and prompt similarity answer different questions, and both depend on the mask you chose.

</details>

## Interpretation and limits · [Evaluation practice]

A LoRA of 1.6 million parameters trained for a few minutes on 36 photographs is compared with the frozen model on twelve held-out photographs, on identical inputs. The denoising loss shows whether the adapter predicts noise better on unseen photographs of the same six birds; preservation shows how much of the kept region survives the MoVQ round trip; CLIP places the outputs between a mean-fill floor and the original photographs. That is the claim, and it is sample-sanity evidence only.

A denoising loss is the training objective, not a quality score. CLIP similarity is a frozen model's opinion, not a human judgement, and CLIP has its own biases about what a species name looks like. Twelve outputs per model from one seeded run give no dispersion estimate. Nothing here measures how natural the boundary between repainted and kept regions looks, the diversity of repaintings, or artefacts. The upstream model may have seen photographs like these during pretraining, which cannot be ruled out. Fine-tuning on a narrow domain can also erode the model elsewhere — the new caption in Section 9 is one sanity check, not a test of generality.

Three things to carry to real data. **Masks are part of the contract:** the model learns to repaint the region you mask, given the rest; a centre mask teaches centred subjects, so use masks shaped like the edits you intend. **Captions describe the whole photograph:** the adapter learns the association between caption text and pixels. **Licences travel with the outputs:** the Kandinsky weights are Apache-2.0 and the photographs here are CC0 — with your own data, the rights to the photographs and to what the adapter produces are yours to establish.

## Conclude with evidence · [Evaluation practice]

Complete this in your own words:

> On [n] held-out photographs of [subject] with [mask], the adapted model's test denoising loss changed from [frozen] to [adapted], its kept-region PSNR from [frozen] to [adapted] dB, and its CLIP prompt similarity from [frozen] to [adapted], between a mean-fill floor of [floor] and an original-photograph ceiling of [ceiling]. Changing the mask size showed [observation]. These are one seeded run on [n] photographs, scored by automated proxies; they do not establish [limitation]. Next I would [specific next step].

<details>
<summary>Check your reasoning (open after answering)</summary>

A strong conclusion names the photographs, the mask and the split; reports the frozen, adapted, floor and ceiling numbers side by side, including any that did not improve; separates what was measured (noise prediction, kept-region fidelity, CLIP agreement) from what was not (how convincing the repainting looks to a person, behaviour on other subjects); and ends with a concrete next step, such as repeating the comparison over several generation seeds or asking people to rate the repainted regions. A weak conclusion says only that "fine-tuning improved inpainting".

</details>

**Transfer:** repeat the notebook with `USE_BYOD = True` on your own photographs, with masks drawn over the regions you actually want to edit, and compare the same four numbers.

## Troubleshooting

| Observation | Appropriate response |
|---|---|
| `'cuda': False` in Section 1 | Runtime → Change runtime type → T4 GPU, then Runtime → Run all. CPU-only runtimes are not supported. |
| The install cell stops with "Core dependencies changed" | Runtime → Restart session, then Run all once more; the pinned versions are then already installed. |
| A download fails, or a size or SHA-256 mismatch is raised | Re-run the cell once for a transient network error. A persistent mismatch means the pinned file changed upstream: stop and report it; do not remove the check. |
| Out of disk | Start a fresh runtime; about 20 GB of free disk is needed for the three snapshots. |
| CUDA out of memory | Use a fresh T4 runtime with the default `BATCH_SIZE = 1`, and make sure Section 5 released the prior. |
| A training loss is `nan` | Keep `LEARNING_RATE` at or below `1e-4` and re-run from Section 3; report the run if it persists. |
| BYOD: "captions.csv is missing columns" or "file ... is not in the dataset" | Fix the named row or file in your zip: `captions.csv` needs `id`, `file`, `caption` (and optionally `mask`), and every named file must be in the zip. |
| BYOD: "mask ... is WxH px but its image is ..." | Save each mask at exactly its photograph's size. |
| BYOD: "split leaves no test record" | Give at least one caption three or more photographs. |
| "this pipeline is already adapted" in Section 6 | Re-run from Section 3 (select it, then Runtime → Run cell and below) so the baseline is the frozen model. |

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can stage and digest-verify three pinned safetensors snapshots, fetch and validate digest-pinned real photographs with their masks, encode captions and release the prior, execute bounded LoRA fine-tuning, evaluate the frozen and the adapted model on identical held-out inputs between a mean-fill floor and an original-photograph ceiling, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or image quality beyond the checks shown.

## References

- Repository README: https://github.com/kurtvalcorza/kandinsky-inpainting-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/kandinsky-inpainting-pipeline/blob/main/MODEL_CARD.md
- Weights notes: https://github.com/kurtvalcorza/kandinsky-inpainting-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face decoder repository: https://huggingface.co/kandinsky-community/kandinsky-2-2-decoder-inpaint (revision `db790ad5cbcabed886f069ef2710774657621702`)
- Hugging Face prior repository: https://huggingface.co/kandinsky-community/kandinsky-2-2-prior (revision 9fc51ad5732afc5d031724219d22e6c42179c5a8)
- Razzhigaev, A., et al. (2023). Kandinsky: An improved text-to-image synthesis with image prior and latent diffusion. arXiv:2310.03502: https://arxiv.org/abs/2310.03502
- Hu, E. J., et al. (2022). LoRA: Low-rank adaptation of large language models. ICLR: https://arxiv.org/abs/2106.09685
- Cherti, M., et al. (2023). Reproducible scaling laws for contrastive language-image learning. CVPR (the LAION CLIP scorer): https://arxiv.org/abs/2212.07143
- Wang, Z., et al. (2004). Image quality assessment: From error visibility to structural similarity. IEEE TIP (SSIM): https://doi.org/10.1109/TIP.2003.819861
- DIMER Notebook Specification 2.2 and Model Card Specification 1.2 (in the ml-worker repository)

**AI assistance disclosure:** generative AI assisted this notebook's code and technical writing under maintainer direction. Tests, pinned identities, manifests and source files remain the authoritative evidence.